# 02. Gradient Descent: El Corazón del Aprendizaje Automático

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 120 minutos (expandido con ejercicios)  
**Prerequisitos:** [01. Regresión Lineal](01-regresion-lineal.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Entender cómo el gradient descent encuentra mínimos de funciones
- **Implementar desde cero** las tres variantes: Batch, Stochastic y Mini-batch GD
- Comprender el impacto crítico del learning rate
- **Aplicar técnicas de optimización avanzadas** (Momentum, RMSprop, Adam)
- Visualizar la superficie de pérdida y el camino de optimización
- **Completar 8 ejercicios prácticos autogradeados** (100 puntos totales)
- Diagnosticar y solucionar problemas comunes de convergencia

<a name='toc'></a>
## 📚 Tabla de Contenidos

- [1 - 📌 Motivación: ¿Por qué Gradient Descent?](#1)
- [2 - 📊 Intuición Visual: Descendiendo por una Montaña](#2)
- [3 - 🧮 Fundamentos Matemáticos](#3)
- [4 - 💻 Implementación Desde Cero](#4)
- [**5 - 🎓 Ejercicios Prácticos Guiados (100 pts)**](#5)
  - [Exercise 1 - `compute_gradient` (10 pts)](#ex-1)
  - [Exercise 2 - `gradient_descent_step` (10 pts)](#ex-2)
  - [Exercise 3 - `batch_gradient_descent` (15 pts)](#ex-3)
  - [Exercise 4 - `create_mini_batches` (10 pts)](#ex-4)
  - [Exercise 5 - `sgd_with_momentum` (15 pts)](#ex-5)
  - [Exercise 6 - `rmsprop_update` (15 pts)](#ex-6)
  - [Exercise 7 - `adam_optimizer` (20 pts)](#ex-7)
  - [Exercise 8 - `learning_rate_decay` (5 pts)](#ex-8)
- [6 - 🏭 Comparación con Frameworks](#6)
- [7 - 🔬 Ejercicios Avanzados](#7)
- [**8 - 📄 Papers y Referencias (15+ papers)**](#8)
- [9 - 📚 Resumen](#9)
- [10 - ➡️ Navegación](#10)

---

**⚡ Nuevo en esta versión:**
- ✅ 8 ejercicios autogradeados estilo Coursera
- ✅ Sistema de puntos (100 pts totales, 70 pts para aprobar)
- ✅ Sección de papers con 15+ referencias específicas
- ✅ Ejercicios avanzados opcionales

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades personalizadas
import sys
sys.path.append('../../shared/utils')
from visualization import plot_loss_surface, plot_optimization_path
from testing import test_exercise, check_shape, check_close
from datasets import load_dataset, generate_synthetic_regression

# Semilla para reproducibilidad
np.random.seed(42)

print("✅ Librerías importadas correctamente")

---
## 📌 1. Motivación: ¿Por qué Gradient Descent?

### El Problema del Mundo Real

Imagina que estás perdido en las montañas durante una noche de niebla densa. Tu objetivo es llegar al valle (el punto más bajo). No puedes ver el paisaje completo, solo puedes:

1. Sentir la pendiente bajo tus pies
2. Dar un paso en la dirección más empinada hacia abajo
3. Repetir hasta llegar al fondo

**¡Esto es exactamente lo que hace Gradient Descent!**

### ¿Por qué es tan importante?

- 🧠 **Es el algoritmo de optimización fundamental** usado en casi todos los modelos de ML
- 🔥 **Entrena las redes neuronales** modernas (GPT, DALL-E, etc.)
- 📈 **Escala a millones de parámetros** (la Normal Equation no puede)
- 🎯 **Es elegante y general** - funciona para cualquier función diferenciable

### Aplicaciones Reales

- 🤖 Entrenamiento de redes neuronales profundas
- 📊 Optimización de modelos de ML en general
- 🎮 Entrenamiento de agentes de Reinforcement Learning
- 💰 Optimización de portafolios financieros
- 🔬 Ajuste de modelos científicos complejos

### La Pregunta Guía

> **¿Cómo puede un algoritmo encontrar el mínimo de una función sin ver toda la superficie, y qué factores determinan su éxito?**

---
## 📊 2. Intuición Visual: Descendiendo por una Montaña

Empecemos visualizando qué es lo que queremos optimizar.

In [ ]:
# Función cuadrática simple en 1D para visualización
def f(x):
    """Función f(x) = (x - 3)^2 + 1"""
    return (x - 3)**2 + 1

def df(x):
    """Derivada: f'(x) = 2(x - 3)"""
    return 2 * (x - 3)

# Crear la curva
x_vals = np.linspace(-2, 8, 200)
y_vals = f(x_vals)

# Simular gradient descent
def gradient_descent_1d(start, learning_rate, n_iterations):
    """Ejecuta gradient descent en 1D"""
    path = [start]
    x = start
    
    for i in range(n_iterations):
        gradient = df(x)
        x = x - learning_rate * gradient
        path.append(x)
    
    return np.array(path)

# Ejecutar con diferentes learning rates
paths = {
    'Muy pequeño (lr=0.05)': gradient_descent_1d(start=7.0, learning_rate=0.05, n_iterations=50),
    'Óptimo (lr=0.3)': gradient_descent_1d(start=7.0, learning_rate=0.3, n_iterations=20),
    'Muy grande (lr=0.9)': gradient_descent_1d(start=7.0, learning_rate=0.9, n_iterations=30)
}

# Visualización
fig = go.Figure()

# Función objetivo
fig.add_trace(go.Scatter(
    x=x_vals,
    y=y_vals,
    mode='lines',
    name='Función f(x) = (x-3)² + 1',
    line=dict(color='lightblue', width=3)
))

# Mínimo verdadero
fig.add_trace(go.Scatter(
    x=[3],
    y=[1],
    mode='markers',
    name='Mínimo Global',
    marker=dict(color='red', size=15, symbol='star')
))

# Paths de optimización
colors = ['orange', 'green', 'purple']
for (name, path), color in zip(paths.items(), colors):
    fig.add_trace(go.Scatter(
        x=path,
        y=f(path),
        mode='lines+markers',
        name=name,
        line=dict(color=color, width=2),
        marker=dict(size=6)
    ))

fig.update_layout(
    title="Gradient Descent con Diferentes Learning Rates",
    xaxis_title="Parámetro x",
    yaxis_title="Costo f(x)",
    template="plotly_white",
    font=dict(size=12),
    height=500
)

fig.show()

print("\n💡 Observaciones Clave:")
print("   • Learning rate pequeño: Progreso lento pero estable")
print("   • Learning rate óptimo: Convergencia rápida y suave")
print("   • Learning rate grande: Oscila y puede divergir")
print("   • El gradiente (pendiente) indica la dirección de mayor aumento")
print("   • Nos movemos en dirección OPUESTA al gradiente (descendiendo)")

In [ ]:
# Visualización 3D: Superficie de pérdida para regresión lineal
# Generar datos simples
np.random.seed(42)
X_simple = 2 * np.random.rand(50, 1)
y_simple = 4 + 3 * X_simple.flatten() + np.random.randn(50)

# Crear grid de parámetros
w_range = np.linspace(0, 6, 50)
b_range = np.linspace(-2, 8, 50)
W, B = np.meshgrid(w_range, b_range)

# Calcular MSE para cada combinación de w y b
MSE = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        w, b = W[i, j], B[i, j]
        y_pred = w * X_simple.flatten() + b
        MSE[i, j] = np.mean((y_simple - y_pred) ** 2)

# Visualización 3D
fig = go.Figure(data=[go.Surface(
    z=MSE,
    x=w_range,
    y=b_range,
    colorscale='Viridis',
    name='Superficie de Pérdida'
)])

fig.update_layout(
    title="Superficie de Pérdida MSE (2 Parámetros)",
    scene=dict(
        xaxis_title='Weight (w)',
        yaxis_title='Bias (b)',
        zaxis_title='MSE',
    ),
    template="plotly_white",
    height=600
)

fig.show()

print("\n💡 Esta es la 'montaña' que Gradient Descent desciende.")
print("   • El punto más bajo es la combinación óptima de w y b")
print("   • El algoritmo solo puede ver la pendiente local, no toda la superficie")

---
## 🧮 3. Fundamentos Matemáticos

### 📖 Notación

| Símbolo | Significado |
|---------|-------------|
| $\theta$ | Parámetros del modelo (puede ser vector) |
| $J(\theta)$ | Función de costo (pérdida) |
| $\nabla_\theta J$ | Gradiente de J respecto a $\theta$ |
| $\alpha$ | Learning rate (tasa de aprendizaje) |
| $t$ | Iteración actual |
| $m$ | Número de ejemplos de entrenamiento |

### El Algoritmo de Gradient Descent

**Idea Central:** Actualizar los parámetros en la dirección opuesta al gradiente.

$$
\begin{align}
\theta^{(t+1)} &= \theta^{(t)} - \alpha \nabla_\theta J(\theta^{(t)}) \tag{1}
\end{align}
$$

Donde:
- $\theta^{(t)}$ son los parámetros en la iteración $t$
- $\alpha$ controla el tamaño del paso
- $\nabla_\theta J$ es el vector de derivadas parciales

### ¿Por qué funciona?

**Teorema de Taylor (aproximación de primer orden):**

$$
J(\theta + \Delta\theta) \approx J(\theta) + \nabla_\theta J \cdot \Delta\theta \tag{2}
$$

Para minimizar $J$, queremos que $J(\theta + \Delta\theta) < J(\theta)$.

Si elegimos $\Delta\theta = -\alpha \nabla_\theta J$:

$$
\begin{align}
J(\theta - \alpha \nabla_\theta J) &\approx J(\theta) - \alpha \|\nabla_\theta J\|^2 \tag{3}\\
&< J(\theta) \quad \text{(si } \alpha \text{ es pequeño)} \tag{4}
\end{align}
$$

**Conclusión:** Moverse en dirección opuesta al gradiente ¡garantiza reducir la pérdida!

### Derivadas para Regresión Lineal

Recordemos la función de costo MSE:

$$
J(w, b) = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2 = \frac{1}{m} \sum_{i=1}^{m} (w^T x^{(i)} + b - y^{(i)})^2 \tag{5}
$$

Las derivadas parciales son:

$$
\begin{align}
\frac{\partial J}{\partial w_j} &= \frac{2}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \cdot x_j^{(i)} \tag{6}\\
\frac{\partial J}{\partial b} &= \frac{2}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \tag{7}
\end{align}
$$

**Nota:** El factor 2 se suele omitir (solo escala el learning rate).

### Ejemplo Numérico

Supongamos:
- 3 ejemplos: $(x, y) = \{(1, 3), (2, 5), (3, 7)\}$
- Parámetros actuales: $w = 1.5$, $b = 1.0$
- Learning rate: $\alpha = 0.1$

**Paso 1: Calcular predicciones**
$$
\begin{align}
\hat{y}^{(1)} &= 1.5(1) + 1.0 = 2.5 \\
\hat{y}^{(2)} &= 1.5(2) + 1.0 = 4.0 \\
\hat{y}^{(3)} &= 1.5(3) + 1.0 = 5.5
\end{align}
$$

**Paso 2: Calcular errores**
$$
\begin{align}
e^{(1)} &= 2.5 - 3 = -0.5 \\
e^{(2)} &= 4.0 - 5 = -1.0 \\
e^{(3)} &= 5.5 - 7 = -1.5
\end{align}
$$

**Paso 3: Calcular gradientes**
$$
\begin{align}
\frac{\partial J}{\partial w} &= \frac{1}{3}[(-0.5)(1) + (-1.0)(2) + (-1.5)(3)] = \frac{-7.0}{3} \approx -2.33 \\
\frac{\partial J}{\partial b} &= \frac{1}{3}[-0.5 - 1.0 - 1.5] = -1.0
\end{align}
$$

**Paso 4: Actualizar parámetros**
$$
\begin{align}
w_{new} &= 1.5 - 0.1(-2.33) = 1.5 + 0.233 = 1.733 \\
b_{new} &= 1.0 - 0.1(-1.0) = 1.0 + 0.1 = 1.1
\end{align}
$$

In [ ]:
# Verificación del ejemplo numérico
X_example = np.array([1, 2, 3])
y_example = np.array([3, 5, 7])
w, b = 1.5, 1.0
alpha = 0.1

# Paso 1: Predicciones
y_pred = w * X_example + b
print("Predicciones:", y_pred)

# Paso 2: Errores
errors = y_pred - y_example
print("Errores:", errors)

# Paso 3: Gradientes
dw = np.mean(errors * X_example)
db = np.mean(errors)
print(f"Gradiente w: {dw:.3f}")
print(f"Gradiente b: {db:.3f}")

# Paso 4: Actualización
w_new = w - alpha * dw
b_new = b - alpha * db
print(f"\nNuevos parámetros:")
print(f"w: {w:.3f} → {w_new:.3f}")
print(f"b: {b:.3f} → {b_new:.3f}")

### Las Tres Variantes de Gradient Descent

#### 1. Batch Gradient Descent

Usa **todos** los datos en cada iteración:

$$
\theta := \theta - \alpha \frac{1}{m} \sum_{i=1}^{m} \nabla_\theta \mathcal{L}(\theta; x^{(i)}, y^{(i)}) \tag{8}
$$

✅ **Ventajas:** Convergencia suave, dirección exacta del gradiente  
❌ **Desventajas:** Lento para datasets grandes

#### 2. Stochastic Gradient Descent (SGD)

Usa **un** ejemplo aleatorio por iteración:

$$
\theta := \theta - \alpha \nabla_\theta \mathcal{L}(\theta; x^{(i)}, y^{(i)}) \tag{9}
$$

✅ **Ventajas:** Muy rápido, puede escapar mínimos locales  
❌ **Desventajas:** Convergencia ruidosa, nunca se estabiliza completamente

#### 3. Mini-batch Gradient Descent

Usa **un subconjunto** (batch) de ejemplos:

$$
\theta := \theta - \alpha \frac{1}{B} \sum_{i=k}^{k+B-1} \nabla_\theta \mathcal{L}(\theta; x^{(i)}, y^{(i)}) \tag{10}
$$

Donde $B$ es el tamaño del batch (típicamente 32, 64, 128, 256).

✅ **Ventajas:** Balance entre velocidad y estabilidad, aprovecha hardware (GPU)  
❌ **Desventajas:** Requiere tunear el batch size

**Comparación:**

| Variante | Ejemplos/iter | Velocidad | Convergencia | Uso |
|----------|---------------|-----------|--------------|-----|
| Batch | $m$ (todos) | Lenta | Suave | Datasets pequeños |
| SGD | 1 | Muy rápida | Ruidosa | Algoritmos online |
| Mini-batch | $B$ (típ. 32-256) | Rápida | Balanceada | **Más usado** |

---
## 💻 4. Implementación Desde Cero

In [ ]:
class GradientDescentOptimizer:
    """
    Implementación completa de Gradient Descent con sus variantes.
    
    Implementa:
    - Batch Gradient Descent
    - Stochastic Gradient Descent (SGD)
    - Mini-batch Gradient Descent
    - Optimizadores avanzados: Momentum, RMSprop, Adam
    
    Parameters:
    -----------
    learning_rate : float
        Tasa de aprendizaje
    n_iterations : int
        Número de iteraciones (epochs)
    batch_size : int or None
        Tamaño del batch. None = Batch GD, 1 = SGD, otro = Mini-batch
    optimizer : str
        'gd', 'momentum', 'rmsprop', 'adam'
    momentum : float
        Parámetro de momentum (beta1)
    beta2 : float
        Parámetro para RMSprop/Adam
    epsilon : float
        Término pequeño para estabilidad numérica
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, batch_size=None,
                 optimizer='gd', momentum=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.batch_size = batch_size
        self.optimizer = optimizer
        self.momentum = momentum
        self.beta2 = beta2
        self.epsilon = epsilon
        
        # Parámetros del modelo
        self.weights = None
        self.bias = None
        
        # Para tracking
        self.losses = []
        self.weight_history = []  # Para visualizar el camino
        
        # Para optimizadores con momento
        self.v_w = None  # Velocidad para weights (momentum)
        self.v_b = None  # Velocidad para bias
        self.s_w = None  # Segundo momento para weights (RMSprop/Adam)
        self.s_b = None  # Segundo momento para bias
        self.t = 0       # Contador de tiempo para Adam
    
    def fit(self, X, y):
        """
        Entrena el modelo usando gradient descent.
        
        Parameters:
        -----------
        X : np.ndarray, shape (n_samples, n_features)
            Features de entrenamiento
        y : np.ndarray, shape (n_samples,)
            Target values
        """
        # Asegurar formato correcto
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        n_samples, n_features = X.shape
        
        # Inicialización de parámetros (pequeños valores aleatorios)
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0.0
        
        # Inicializar momentos
        self.v_w = np.zeros_like(self.weights)
        self.v_b = 0.0
        self.s_w = np.zeros_like(self.weights)
        self.s_b = 0.0
        
        # Determinar batch size
        if self.batch_size is None:
            batch_size = n_samples  # Batch GD
        else:
            batch_size = min(self.batch_size, n_samples)
        
        print(f"\n🏃 Iniciando entrenamiento...")
        print(f"   Optimizador: {self.optimizer}")
        print(f"   Batch size: {batch_size} {'(Batch GD)' if batch_size == n_samples else '(Mini-batch)' if batch_size > 1 else '(SGD)'}")
        print(f"   Learning rate: {self.lr}")
        print()
        
        # Training loop
        for epoch in range(self.n_iter):
            # Shuffle de datos (importante para SGD/Mini-batch)
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            # Procesar en batches
            epoch_loss = 0
            n_batches = 0
            
            for i in range(0, n_samples, batch_size):
                # Obtener batch
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                batch_samples = len(X_batch)
                
                # Forward pass
                y_pred = X_batch @ self.weights + self.bias
                
                # Calcular loss
                loss = np.mean((y_batch - y_pred) ** 2)
                epoch_loss += loss
                n_batches += 1
                
                # Calcular gradientes
                error = y_pred - y_batch
                dw = (1 / batch_samples) * (X_batch.T @ error)
                db = (1 / batch_samples) * np.sum(error)
                
                # Actualizar parámetros según el optimizador
                self._update_parameters(dw, db)
            
            # Promedio de loss de la época
            avg_loss = epoch_loss / n_batches
            self.losses.append(avg_loss)
            
            # Guardar historial de weights para visualización
            if n_features <= 2:  # Solo para visualización 2D/3D
                self.weight_history.append((self.weights.copy(), self.bias))
            
            # Logging
            if epoch % max(1, self.n_iter // 10) == 0:
                print(f"Epoch {epoch:4d} - Loss: {avg_loss:.6f}")
        
        print(f"\n✅ Entrenamiento completado")
        print(f"   Loss final: {self.losses[-1]:.6f}")
        print(f"   Weights: {self.weights}")
        print(f"   Bias: {self.bias:.6f}")
        
        return self
    
    def _update_parameters(self, dw, db):
        """
        Actualiza parámetros según el optimizador elegido.
        """
        self.t += 1  # Incrementar contador de tiempo
        
        if self.optimizer == 'gd':
            # Vanilla Gradient Descent
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
        
        elif self.optimizer == 'momentum':
            # Gradient Descent con Momentum
            # v = beta * v + (1 - beta) * gradient
            # theta = theta - lr * v
            self.v_w = self.momentum * self.v_w + (1 - self.momentum) * dw
            self.v_b = self.momentum * self.v_b + (1 - self.momentum) * db
            
            self.weights -= self.lr * self.v_w
            self.bias -= self.lr * self.v_b
        
        elif self.optimizer == 'rmsprop':
            # RMSprop: Divide learning rate por raíz del promedio de gradientes al cuadrado
            # s = beta2 * s + (1 - beta2) * gradient^2
            # theta = theta - lr * gradient / sqrt(s + epsilon)
            self.s_w = self.beta2 * self.s_w + (1 - self.beta2) * (dw ** 2)
            self.s_b = self.beta2 * self.s_b + (1 - self.beta2) * (db ** 2)
            
            self.weights -= self.lr * dw / (np.sqrt(self.s_w) + self.epsilon)
            self.bias -= self.lr * db / (np.sqrt(self.s_b) + self.epsilon)
        
        elif self.optimizer == 'adam':
            # Adam: Combina Momentum y RMSprop
            # m = beta1 * m + (1 - beta1) * gradient (primer momento)
            # v = beta2 * v + (1 - beta2) * gradient^2 (segundo momento)
            # m_corrected = m / (1 - beta1^t) (corrección de sesgo)
            # v_corrected = v / (1 - beta2^t)
            # theta = theta - lr * m_corrected / (sqrt(v_corrected) + epsilon)
            
            # Actualizar momentos
            self.v_w = self.momentum * self.v_w + (1 - self.momentum) * dw
            self.v_b = self.momentum * self.v_b + (1 - self.momentum) * db
            self.s_w = self.beta2 * self.s_w + (1 - self.beta2) * (dw ** 2)
            self.s_b = self.beta2 * self.s_b + (1 - self.beta2) * (db ** 2)
            
            # Corrección de sesgo
            v_w_corrected = self.v_w / (1 - self.momentum ** self.t)
            v_b_corrected = self.v_b / (1 - self.momentum ** self.t)
            s_w_corrected = self.s_w / (1 - self.beta2 ** self.t)
            s_b_corrected = self.s_b / (1 - self.beta2 ** self.t)
            
            # Actualización
            self.weights -= self.lr * v_w_corrected / (np.sqrt(s_w_corrected) + self.epsilon)
            self.bias -= self.lr * v_b_corrected / (np.sqrt(s_b_corrected) + self.epsilon)
    
    def predict(self, X):
        """Hace predicciones"""
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        return X @ self.weights + self.bias
    
    def score(self, X, y):
        """Calcula R²"""
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

print("✅ Clase GradientDescentOptimizer definida")

### Probemos las diferentes variantes

In [ ]:
# Generar datos de prueba
X_train, X_test, y_train, y_test = generate_synthetic_regression(
    n_samples=1000,
    n_features=1,
    noise=10.0,
    random_state=42
)

print(f"📊 Datos generados: {X_train.shape[0]} ejemplos de entrenamiento")

In [ ]:
# Comparar las tres variantes básicas
variants = [
    ('Batch GD', None),
    ('Mini-batch GD', 32),
    ('SGD', 1)
]

results = {}

for name, batch_size in variants:
    print(f"\n{'='*60}")
    print(f" {name}")
    print(f"{'='*60}")
    
    model = GradientDescentOptimizer(
        learning_rate=0.01,
        n_iterations=100,
        batch_size=batch_size,
        optimizer='gd'
    )
    model.fit(X_train, y_train)
    
    r2 = model.score(X_test, y_test)
    results[name] = {
        'model': model,
        'r2': r2
    }
    print(f"   R² en test: {r2:.4f}")

In [ ]:
# Visualizar convergencia de las tres variantes
fig = go.Figure()

colors = ['blue', 'green', 'red']
for (name, _), color in zip(variants, colors):
    losses = results[name]['model'].losses
    fig.add_trace(go.Scatter(
        y=losses,
        mode='lines',
        name=name,
        line=dict(color=color, width=2)
    ))

fig.update_layout(
    title="Comparación de Convergencia: Batch vs Mini-batch vs SGD",
    xaxis_title="Época",
    yaxis_title="Loss (MSE)",
    template="plotly_white",
    font=dict(size=12),
    height=500
)

fig.show()

print("\n💡 Observaciones:")
print("   • Batch GD: Convergencia suave y estable")
print("   • Mini-batch: Balance entre velocidad y estabilidad")
print("   • SGD: Convergencia ruidosa pero puede explorar más")

### Comparación de Optimizadores Avanzados

In [ ]:
# Comparar optimizadores
optimizers = ['gd', 'momentum', 'rmsprop', 'adam']
opt_results = {}

for opt in optimizers:
    print(f"\n🔧 Entrenando con {opt.upper()}...")
    
    model = GradientDescentOptimizer(
        learning_rate=0.01,
        n_iterations=200,
        batch_size=32,
        optimizer=opt
    )
    model.fit(X_train, y_train)
    
    r2 = model.score(X_test, y_test)
    opt_results[opt] = {
        'model': model,
        'r2': r2
    }

# Visualizar
fig = go.Figure()

colors = ['blue', 'orange', 'green', 'red']
for opt, color in zip(optimizers, colors):
    losses = opt_results[opt]['model'].losses
    r2 = opt_results[opt]['r2']
    fig.add_trace(go.Scatter(
        y=losses,
        mode='lines',
        name=f'{opt.upper()} (R²={r2:.4f})',
        line=dict(color=color, width=2)
    ))

fig.update_layout(
    title="Comparación de Optimizadores",
    xaxis_title="Época",
    yaxis_title="Loss (MSE)",
    template="plotly_white",
    font=dict(size=12),
    height=500
)

fig.show()

print("\n📊 Comparación de Optimizadores:")
print("\n" + "="*60)
print(f"{'Optimizador':<15} {'R² Test':<10} {'Loss Final':<15}")
print("="*60)
for opt in optimizers:
    r2 = opt_results[opt]['r2']
    loss = opt_results[opt]['model'].losses[-1]
    print(f"{opt.upper():<15} {r2:<10.4f} {loss:<15.6f}")
print("="*60)

print("\n💡 Adam típicamente converge más rápido y de forma más estable.")

---
## 🏭 5. Versión con Framework (PyTorch/TensorFlow)

Los frameworks modernos implementan estos optimizadores de forma altamente optimizada.

In [ ]:
# Ejemplo con TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras

# Crear modelo simple
model = keras.Sequential([
    keras.layers.Dense(1, input_shape=(1,))
])

# Configurar diferentes optimizadores
tf_optimizers = {
    'SGD': keras.optimizers.SGD(learning_rate=0.01),
    'SGD+Momentum': keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'RMSprop': keras.optimizers.RMSprop(learning_rate=0.01),
    'Adam': keras.optimizers.Adam(learning_rate=0.01)
}

tf_results = {}

for name, optimizer in tf_optimizers.items():
    print(f"\n🔧 Entrenando con {name}...")
    
    # Compilar modelo
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    # Entrenar
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_test, y_test),
        verbose=0
    )
    
    # Evaluar
    test_loss = model.evaluate(X_test, y_test, verbose=0)
    tf_results[name] = {
        'history': history.history,
        'test_loss': test_loss
    }
    print(f"   Loss final en test: {test_loss[0]:.6f}")

print("\n✅ Frameworks como TensorFlow/PyTorch ofrecen:")
print("   • Implementaciones altamente optimizadas (C++/CUDA)")
print("   • Diferenciación automática (no calculas gradientes manualmente)")
print("   • Soporte para GPU aceleración")
print("   • Muchos optimizadores adicionales (AdaGrad, AdaDelta, etc.)")

---
## 🎯 6. Ejercicios Prácticos

### 🟢 Ejercicio 1: Efecto del Learning Rate

Experimenta con diferentes learning rates y observa el efecto en la convergencia.

In [ ]:
def ejercicio_1():
    """
    Objetivo: Entender el impacto del learning rate
    
    Instrucciones:
    1. Entrena 3 modelos con learning_rates: [0.001, 0.01, 0.1]
    2. Usa batch_size=32, n_iterations=200
    3. Grafica las curvas de loss
    4. Retorna el learning rate que converge más rápido
    
    Returns:
    --------
    best_lr : float
        El learning rate óptimo
    """
    # TODO: Tu código aquí
    # learning_rates = [0.001, 0.01, 0.1]
    # for lr in learning_rates:
    #     model = GradientDescentOptimizer(learning_rate=lr, ...)
    #     ...
    
    pass

# Descomentar para probar
# best_lr = ejercicio_1()
# print(f"\n📊 Mejor learning rate: {best_lr}")

### 🟡 Ejercicio 2: Implementar Early Stopping

El early stopping detiene el entrenamiento cuando la pérdida en validación deja de mejorar.

In [ ]:
def ejercicio_2():
    """
    Objetivo: Implementar early stopping
    
    Instrucciones:
    1. Modifica GradientDescentOptimizer para aceptar un conjunto de validación
    2. En cada época, calcula loss en validación
    3. Si el loss no mejora en 'patience' épocas, detén el entrenamiento
    4. Restaura los mejores pesos encontrados
    
    Pistas:
    - Guarda best_loss y best_weights
    - Usa un contador para patience
    - Compara loss_val con best_loss en cada época
    
    Returns:
    --------
    epoch_stopped : int
        Época en la que se detuvo el entrenamiento
    """
    # TODO: Tu código aquí
    
    pass

# Descomentar para probar
# epoch = ejercicio_2()
# print(f"\n📊 Entrenamiento detenido en época: {epoch}")

### 🔴 Ejercicio 3: Learning Rate Scheduling

Implementa un schedule que reduzca el learning rate durante el entrenamiento.

In [ ]:
def ejercicio_3():
    """
    Objetivo: Implementar learning rate decay
    
    Instrucciones:
    1. Implementa 3 estrategias de decay:
       a) Step decay: lr = lr * decay_rate cada N épocas
       b) Exponential decay: lr = lr_initial * e^(-decay_rate * epoch)
       c) 1/t decay: lr = lr_initial / (1 + decay_rate * epoch)
    2. Compara las tres estrategias
    3. Grafica learning rate vs época y loss vs época
    
    Returns:
    --------
    dict : {'step': losses, 'exponential': losses, '1/t': losses}
    """
    # TODO: Tu código aquí
    # Pista: Modifica el learning rate en cada época
    # if epoch % step_size == 0:
    #     self.lr *= decay_rate
    
    pass

# Descomentar para probar
# results = ejercicio_3()
# print("\n📊 Comparación de estrategias de decay completada")

---
## 📚 7. Resumen y Recursos

### 🎯 Puntos Clave

1. **Gradient Descent** es el algoritmo de optimización fundamental en ML, usado para minimizar funciones de pérdida

2. **La regla de actualización** es simple pero poderosa: $\theta := \theta - \alpha \nabla_\theta J(\theta)$

3. **Tres variantes principales**:
   - Batch GD: Usa todos los datos (estable pero lento)
   - SGD: Usa un ejemplo (rápido pero ruidoso)
   - Mini-batch GD: Balance óptimo (más usado en práctica)

4. **El learning rate es crítico**:
   - Muy pequeño: Convergencia muy lenta
   - Muy grande: Oscilaciones o divergencia
   - Típicamente se usa: 0.001, 0.01, 0.1 (probar varios)

5. **Optimizadores avanzados** mejoran la convergencia:
   - **Momentum**: Acelera en direcciones consistentes
   - **RMSprop**: Adapta learning rate por parámetro
   - **Adam**: Combina momentum + RMSprop (más popular)

6. **En producción** usa frameworks (TensorFlow, PyTorch) que implementan estos algoritmos de forma altamente optimizada

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

- **"Adam: A Method for Stochastic Optimization"** - Kingma & Ba (2014)
  - [Link al paper](https://arxiv.org/abs/1412.6980)
  - Introduce el optimizador Adam, ahora el más usado en deep learning

- **"On the importance of initialization and momentum in deep learning"** - Sutskever et al. (2013)
  - Explica por qué momentum es tan efectivo

#### 📖 Recursos Interactivos

- **Distill.pub - "Why Momentum Really Works"**
  - [https://distill.pub/2017/momentum/](https://distill.pub/2017/momentum/)
  - Visualizaciones interactivas excelentes

- **CS231n - Optimization Notes**
  - [http://cs231n.github.io/optimization-1/](http://cs231n.github.io/optimization-1/)
  - Explicación detallada con visualizaciones

#### 🎥 Videos Recomendados

- **Andrew Ng - Gradient Descent Lectures**
  - Coursera Machine Learning Course
  - Explicación clara y paso a paso

- **3Blue1Brown - Gradient Descent**
  - Intuición geométrica hermosa

#### 💻 Implementaciones de Referencia

- [PyTorch Optimizers](https://pytorch.org/docs/stable/optim.html)
- [TensorFlow Optimizers](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers)
- [JAX Optimizers (Optax)](https://github.com/deepmind/optax)

### 🤔 Preguntas para Reflexionar

1. **¿Por qué Adam funciona tan bien en la práctica?**
   - Combina lo mejor de momentum y adaptive learning rates
   - Funciona bien con valores por defecto

2. **¿Cuándo usar SGD vs Adam?**
   - SGD + momentum puede generalizar mejor (menos overfitting)
   - Adam converge más rápido
   - Depende del problema y arquitectura

3. **¿Cómo escalar el learning rate con el batch size?**
   - Regla común: lr_new = lr_base * (batch_size_new / batch_size_base)
   - Ver "Linear Scaling Rule" de Goyal et al. (2017)

### 📊 Tabla de Referencia Rápida

| Optimizador | Pros | Contras | Cuándo usar |
|-------------|------|---------|-------------|
| **SGD** | Simple, bien entendido | Requiere tunear LR | Cuando tienes tiempo |
| **SGD + Momentum** | Acelera convergencia | Aún requiere tunear | Recomendado para CV |
| **Adam** | Funciona out-of-the-box | Puede overfittear | Primera opción (default) |
| **RMSprop** | Bueno para RNNs | Menos usado ahora | Problemas recurrentes |

---

## ➡️ Próximo Paso

En el siguiente notebook, **03. Regresión Logística**, aplicaremos gradient descent a nuestro primer problema de **clasificación**:

- Pasar de regresión (predecir valores continuos) a clasificación (predecir categorías)
- La función sigmoide para probabilidades
- Cross-entropy como función de pérdida
- Interpretación probabilística
- Métricas de clasificación (accuracy, precision, recall)

---

<div align="center">

**🎉 ¡Has dominado el corazón del Machine Learning! 🎉**

**Continúa con: [03. Regresión Logística](03-regresion-logistica.ipynb)**

[← 01. Regresión Lineal](01-regresion-lineal.ipynb) | [Índice de ML Clásico](README.md) | [03. Regresión Logística →](03-regresion-logistica.ipynb)

</div>

<a name='5'></a>

---
## 🎓 5. Ejercicios Prácticos Guiados (100 pts)

Esta sección contiene **8 ejercicios autogradeados** que te guiarán paso a paso en la implementación de gradient descent y sus optimizadores avanzados.

### 📊 Sistema de Puntos

| Ejercicio | Función | Puntos | Dificultad | Tiempo |
|-----------|---------|--------|------------|--------|
| 1 | `compute_gradient` | 10 | 🟢 Fácil | 8 min |
| 2 | `gradient_descent_step` | 10 | 🟢 Fácil | 8 min |
| 3 | `batch_gradient_descent` | 15 | 🟡 Medio | 12 min |
| 4 | `create_mini_batches` | 10 | 🟢 Fácil | 8 min |
| 5 | `sgd_with_momentum` | 15 | 🟡 Medio | 12 min |
| 6 | `rmsprop_update` | 15 | 🟡 Medio | 12 min |
| 7 | `adam_optimizer` | 20 | 🔴 Difícil | 15 min |
| 8 | `learning_rate_decay` | 5 | 🟢 Fácil | 5 min |
| **TOTAL** | | **100** | | **80 min** |

**Mínimo para aprobar:** 70 puntos

### 🎯 Instrucciones

1. **Lee el docstring** de cada función cuidadosamente
2. **Implementa el código** entre `# YOUR CODE STARTS HERE` y `# YOUR CODE ENDS HERE`
3. **Ejecuta la celda de test** inmediatamente después de cada función
4. **Verifica los resultados** - el autograder te dirá si pasaste todos los tests
5. **Obtén al menos 70 puntos** para aprobar el notebook

### 🚀 Importar el Autograder

In [ ]:
# Importar sistema de autograding
import sys
sys.path.append('./tests')
from test_02_gradient_descent import GradientDescentGrader

# Inicializar grader
grader = GradientDescentGrader()

print("✅ Autograder importado correctamente")
print(f"📊 Puntos totales disponibles: {grader.total_points}")
print(f"📊 Puntos mínimos para aprobar: 70")
print("\n🎯 ¡Comencemos con los ejercicios!")

<a name='ex-1'></a>

---
### Exercise 1 - compute_gradient

**GRADED FUNCTION: `compute_gradient`**

**Dificultad:** 🟢 Fácil | **Puntos:** 10 | **Tiempo estimado:** 8 min

#### Objetivo

Implementa la función que calcula el gradiente de la función de costo MSE con respecto a los pesos y el bias.

#### Teoría

Para la función de costo MSE:

$$J(w, b) = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2$$

Las derivadas parciales son:

$$\frac{\partial J}{\partial w} = \frac{2}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \cdot x^{(i)}$$

$$\frac{\partial J}{\partial b} = \frac{2}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$

Donde $\hat{y}^{(i)} = w \cdot x^{(i)} + b$

#### Instrucciones

Implementa `compute_gradient(X, y, w, b)` que retorna `(dw, db)`:
1. Calcula las predicciones $\hat{y}$
2. Calcula el error $e = \hat{y} - y$
3. Calcula el gradiente de los pesos: `dw`
4. Calcula el gradiente del bias: `db`

**Nota:** Puedes omitir el factor 2 (solo escala el learning rate).

In [ ]:
# GRADED FUNCTION: compute_gradient

def compute_gradient(X, y, w, b):
    """
    Calcula el gradiente de la función de costo MSE.
    
    Arguments:
    X -- array de features, shape (m, n)
    y -- array de targets, shape (m,)
    w -- array de pesos, shape (n,)
    b -- bias, escalar
    
    Returns:
    dw -- gradiente de los pesos, shape (n,)
    db -- gradiente del bias, escalar
    """
    
    m = X.shape[0]  # número de ejemplos
    
    # Step 1: Calcular predicciones
    # (approx. 1 line)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # Step 2: Calcular error
    # (approx. 1 line)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # Step 3: Calcular gradiente de los pesos
    # Hint: usa np.dot() o @, divide por m
    # (approx. 1 line)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # Step 4: Calcular gradiente del bias
    # (approx. 1 line)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return dw, db

In [ ]:
# Test Exercise 1
grader.test_compute_gradient(compute_gradient)

<a name='ex-2'></a>

---
### Exercise 2 - gradient_descent_step

**GRADED FUNCTION: `gradient_descent_step`**

**Dificultad:** 🟢 Fácil | **Puntos:** 10 | **Tiempo estimado:** 8 min

#### Objetivo

Implementa un paso de gradient descent que actualiza los parámetros.

#### Teoría

La regla de actualización de gradient descent es:

$$w := w - \alpha \frac{\partial J}{\partial w}$$
$$b := b - \alpha \frac{\partial J}{\partial b}$$

Donde $\alpha$ es el learning rate.

#### Instrucciones

Implementa `gradient_descent_step(w, b, dw, db, learning_rate)` que retorna `(w_new, b_new)`:
1. Actualiza los pesos usando la regla de GD
2. Actualiza el bias usando la regla de GD

In [ ]:
# GRADED FUNCTION: gradient_descent_step

def gradient_descent_step(w, b, dw, db, learning_rate):
    """
    Realiza un paso de gradient descent.
    
    Arguments:
    w -- pesos actuales, shape (n,)
    b -- bias actual, escalar
    dw -- gradiente de los pesos, shape (n,)
    db -- gradiente del bias, escalar
    learning_rate -- tasa de aprendizaje, escalar
    
    Returns:
    w_new -- pesos actualizados, shape (n,)
    b_new -- bias actualizado, escalar
    """
    
    # Step 1: Actualizar pesos
    # (approx. 1 line)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # Step 2: Actualizar bias
    # (approx. 1 line)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return w_new, b_new

In [ ]:
# Test Exercise 2
grader.test_gradient_descent_step(gradient_descent_step)

<a name='ex-3'></a>

---
### Exercise 3 - batch_gradient_descent

**GRADED FUNCTION: `batch_gradient_descent`**

**Dificultad:** 🟡 Medio | **Puntos:** 15 | **Tiempo estimado:** 12 min

#### Objetivo

Implementa el algoritmo completo de Batch Gradient Descent.

#### Teoría

Batch GD usa **todos** los ejemplos de entrenamiento en cada iteración:

**Algoritmo:**
```
Inicializar w, b
Para cada época:
    1. Calcular gradiente usando TODOS los datos
    2. Actualizar parámetros
    3. Guardar loss
```

#### Instrucciones

Implementa `batch_gradient_descent(X, y, learning_rate, n_iterations)` que retorna `(w, b, losses)`:
1. Inicializa w con ceros y b = 0
2. Para cada iteración:
   - Calcula el gradiente usando `compute_gradient`
   - Actualiza los parámetros usando `gradient_descent_step`
   - Calcula y guarda el MSE loss
3. Retorna parámetros finales y lista de losses

In [ ]:
# GRADED FUNCTION: batch_gradient_descent

def batch_gradient_descent(X, y, learning_rate, n_iterations):
    """
    Implementa Batch Gradient Descent completo.
    
    Arguments:
    X -- features, shape (m, n)
    y -- targets, shape (m,)
    learning_rate -- tasa de aprendizaje
    n_iterations -- número de iteraciones
    
    Returns:
    w -- pesos finales, shape (n,)
    b -- bias final, escalar
    losses -- lista de losses por iteración
    """
    
    m, n = X.shape
    
    # Step 1: Inicializar parámetros
    # (approx. 2 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    losses = []
    
    # Step 2: Loop de entrenamiento
    for i in range(n_iterations):
        
        # Step 2a: Calcular gradiente
        # (approx. 1 line)
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
        # Step 2b: Actualizar parámetros
        # (approx. 1 line)
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
        # Step 2c: Calcular loss (MSE)
        # (approx. 2-3 lines)
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
    
    return w, b, losses

In [ ]:
# Test Exercise 3
grader.test_batch_gradient_descent(batch_gradient_descent)

<a name='ex-4'></a>

---
### Exercise 4 - create_mini_batches

**GRADED FUNCTION: `create_mini_batches`**

**Dificultad:** 🟢 Fácil | **Puntos:** 10 | **Tiempo estimado:** 8 min

#### Objetivo

Crea mini-batches para entrenamiento.

#### Teoría

Mini-batch GD divide el dataset en pequeños grupos (batches) y actualiza los parámetros después de cada batch.

**Ventajas:**
- Más rápido que Batch GD
- Más estable que SGD
- Aprovecha paralelización en GPU

#### Instrucciones

Implementa `create_mini_batches(X, y, batch_size)` que retorna una lista de tuplas `(X_batch, y_batch)`:
1. Shuffle los datos (usa un permutation aleatorio)
2. Divide en batches de tamaño `batch_size`
3. El último batch puede ser más pequeño

In [ ]:
# GRADED FUNCTION: create_mini_batches

def create_mini_batches(X, y, batch_size, seed=0):
    """
    Crea mini-batches del dataset.
    
    Arguments:
    X -- features, shape (m, n)
    y -- targets, shape (m,)
    batch_size -- tamaño de cada batch
    seed -- semilla para reproducibilidad
    
    Returns:
    mini_batches -- lista de tuplas (X_batch, y_batch)
    """
    
    np.random.seed(seed)
    m = X.shape[0]
    mini_batches = []
    
    # Step 1: Shuffle
    # (approx. 3 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # Step 2: Particionar en batches
    # (approx. 4-5 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return mini_batches

In [ ]:
# Test Exercise 4
grader.test_create_mini_batches(create_mini_batches)

<a name='ex-5'></a>

---
### Exercise 5 - sgd_with_momentum

**GRADED FUNCTION: `sgd_with_momentum`**

**Dificultad:** 🟡 Medio | **Puntos:** 15 | **Tiempo estimado:** 12 min

#### Objetivo

Implementa Momentum para acelerar la convergencia.

#### Teoría

Momentum acelera el gradiente descendente acumulando velocidad en direcciones consistentes:

$$v_t = \beta v_{t-1} + (1 - \beta) \nabla J$$
$$\theta_t = \theta_{t-1} - \alpha v_t$$

Donde:
- $v$ es la "velocidad" (exponential moving average del gradiente)
- $\beta$ es el coeficiente de momentum (típicamente 0.9)
- $\alpha$ es el learning rate

**Ventajas:**
- Acelera en direcciones consistentes
- Reduce oscilaciones
- Puede escapar mínimos locales

#### Instrucciones

Implementa la actualización de parámetros con momentum:
1. Actualiza la velocidad v usando la fórmula de momentum
2. Actualiza los parámetros usando la velocidad

In [ ]:
# GRADED FUNCTION: sgd_with_momentum

def sgd_with_momentum(w, b, dw, db, v_w, v_b, learning_rate, beta=0.9):
    """
    Actualiza parámetros usando momentum.
    
    Arguments:
    w -- pesos, shape (n,)
    b -- bias, escalar
    dw -- gradiente de w, shape (n,)
    db -- gradiente de b, escalar
    v_w -- velocidad de w, shape (n,)
    v_b -- velocidad de b, escalar
    learning_rate -- tasa de aprendizaje
    beta -- coeficiente de momentum
    
    Returns:
    w -- pesos actualizados
    b -- bias actualizado
    v_w -- velocidad actualizada de w
    v_b -- velocidad actualizada de b
    """
    
    # Step 1: Actualizar velocidades
    # v = beta * v + (1 - beta) * gradient
    # (approx. 2 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # Step 2: Actualizar parámetros
    # theta = theta - lr * v
    # (approx. 2 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return w, b, v_w, v_b

In [ ]:
# Test Exercise 5
grader.test_sgd_with_momentum(sgd_with_momentum)

<a name='ex-6'></a>

---
### Exercise 6 - rmsprop_update

**GRADED FUNCTION: `rmsprop_update`**

**Dificultad:** 🟡 Medio | **Puntos:** 15 | **Tiempo estimado:** 12 min

#### Objetivo

Implementa RMSprop, que adapta el learning rate por parámetro.

#### Teoría

RMSprop (Root Mean Square Propagation) divide el learning rate por la raíz del promedio móvil exponencial de los gradientes al cuadrado:

$$s_t = \beta_2 s_{t-1} + (1 - \beta_2) (\nabla J)^2$$
$$\theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{s_t + \epsilon}} \nabla J$$

Donde:
- $s$ es el segundo momento (promedio de gradientes al cuadrado)
- $\beta_2$ típicamente 0.999
- $\epsilon$ es un término pequeño para estabilidad numérica (1e-8)

**Ventaja:** Parámetros con gradientes grandes reciben updates más pequeños.

#### Instrucciones

Implementa la actualización RMSprop:
1. Actualiza s usando exponential moving average de gradientes al cuadrado
2. Actualiza parámetros dividiendo por sqrt(s + epsilon)

In [ ]:
# GRADED FUNCTION: rmsprop_update

def rmsprop_update(w, b, dw, db, s_w, s_b, learning_rate, beta2=0.999, epsilon=1e-8):
    """
    Actualiza parámetros usando RMSprop.
    
    Arguments:
    w -- pesos, shape (n,)
    b -- bias, escalar
    dw -- gradiente de w, shape (n,)
    db -- gradiente de b, escalar
    s_w -- segundo momento de w, shape (n,)
    s_b -- segundo momento de b, escalar
    learning_rate -- tasa de aprendizaje
    beta2 -- coeficiente para segundo momento
    epsilon -- término para estabilidad
    
    Returns:
    w -- pesos actualizados
    b -- bias actualizado
    s_w -- segundo momento actualizado de w
    s_b -- segundo momento actualizado de b
    """
    
    # Step 1: Actualizar segundos momentos
    # s = beta2 * s + (1 - beta2) * gradient^2
    # (approx. 2 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # Step 2: Actualizar parámetros
    # theta = theta - lr * gradient / sqrt(s + epsilon)
    # (approx. 2 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return w, b, s_w, s_b

In [ ]:
# Test Exercise 6
grader.test_rmsprop_update(rmsprop_update)

<a name='ex-7'></a>

---
### Exercise 7 - adam_optimizer

**GRADED FUNCTION: `adam_optimizer`**

**Dificultad:** 🔴 Difícil | **Puntos:** 20 | **Tiempo estimado:** 15 min

#### Objetivo

Implementa Adam, el optimizador más popular en Deep Learning.

#### Teoría

Adam (Adaptive Moment Estimation) combina Momentum y RMSprop:

**Paso 1: Actualizar momentos**
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla J \quad \text{(primer momento)}$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) (\nabla J)^2 \quad \text{(segundo momento)}$$

**Paso 2: Corrección de sesgo**
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$
$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

**Paso 3: Actualizar parámetros**
$$\theta_t = \theta_{t-1} - \frac{\alpha \hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

Donde:
- $\beta_1 = 0.9$ (momentum)
- $\beta_2 = 0.999$ (RMSprop)
- $t$ es el paso de tiempo (empieza en 1)
- La corrección de sesgo es crucial al inicio

**Por qué Adam es tan popular:**
- Combina lo mejor de Momentum y RMSprop
- Funciona bien con hiperparámetros por defecto
- Adaptativo por parámetro
- Robusto en práctica

#### Instrucciones

Implementa Adam completo:
1. Actualiza primer momento (m)
2. Actualiza segundo momento (v)
3. Aplica corrección de sesgo
4. Actualiza parámetros

In [ ]:
# GRADED FUNCTION: adam_optimizer

def adam_optimizer(w, b, dw, db, m_w, m_b, v_w, v_b, t, learning_rate, 
                   beta1=0.9, beta2=0.999, epsilon=1e-8):
    """
    Actualiza parámetros usando Adam.
    
    Arguments:
    w -- pesos, shape (n,)
    b -- bias, escalar
    dw -- gradiente de w, shape (n,)
    db -- gradiente de b, escalar
    m_w -- primer momento de w, shape (n,)
    m_b -- primer momento de b, escalar
    v_w -- segundo momento de w, shape (n,)
    v_b -- segundo momento de b, escalar
    t -- paso de tiempo (empieza en 1)
    learning_rate -- tasa de aprendizaje
    beta1 -- coeficiente para primer momento
    beta2 -- coeficiente para segundo momento
    epsilon -- término para estabilidad
    
    Returns:
    w -- pesos actualizados
    b -- bias actualizado
    m_w, m_b -- primer momento actualizado
    v_w, v_b -- segundo momento actualizado
    """
    
    # Step 1: Actualizar primer momento (momentum)
    # m = beta1 * m + (1 - beta1) * gradient
    # (approx. 2 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # Step 2: Actualizar segundo momento (RMSprop)
    # v = beta2 * v + (1 - beta2) * gradient^2
    # (approx. 2 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # Step 3: Corrección de sesgo
    # m_corrected = m / (1 - beta1^t)
    # v_corrected = v / (1 - beta2^t)
    # (approx. 4 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # Step 4: Actualizar parámetros
    # theta = theta - lr * m_corrected / (sqrt(v_corrected) + epsilon)
    # (approx. 2 lines)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return w, b, m_w, m_b, v_w, v_b

In [ ]:
# Test Exercise 7
grader.test_adam_optimizer(adam_optimizer)

<a name='ex-8'></a>

---
### Exercise 8 - learning_rate_decay

**GRADED FUNCTION: `learning_rate_decay`**

**Dificultad:** 🟢 Fácil | **Puntos:** 5 | **Tiempo estimado:** 5 min

#### Objetivo

Implementa decaimiento exponencial del learning rate.

#### Teoría

Reducir el learning rate durante el entrenamiento puede mejorar la convergencia:

$$\alpha_t = \alpha_0 \cdot e^{-k \cdot t}$$

Donde:
- $\alpha_0$ es el learning rate inicial
- $k$ es la tasa de decay
- $t$ es el número de época

**Intuición:** Al inicio damos pasos grandes para explorar, al final pasos pequeños para afinar.

#### Instrucciones

Implementa `learning_rate_decay(lr_initial, epoch, decay_rate)`.

In [ ]:
# GRADED FUNCTION: learning_rate_decay

def learning_rate_decay(lr_initial, epoch, decay_rate):
    """
    Calcula el learning rate con decaimiento exponencial.
    
    Arguments:
    lr_initial -- learning rate inicial
    epoch -- número de época actual
    decay_rate -- tasa de decaimiento (k)
    
    Returns:
    lr -- learning rate actualizado
    """
    
    # Fórmula: lr = lr_initial * exp(-decay_rate * epoch)
    # Hint: usa np.exp()
    # (approx. 1 line)
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return lr

In [ ]:
# Test Exercise 8
grader.test_learning_rate_decay(learning_rate_decay)

In [ ]:
# Obtener calificación final
grader.get_grade_summary()

<a name='8'></a>

---
## 📄 8. Papers y Referencias Específicas

Esta sección contiene **15+ papers fundamentales** sobre gradient descent y optimización, organizados por categoría.

### 📚 Papers Seminales (Fundamentos)

#### 1. **Robbins & Monro (1951)** - El Origen del SGD
**"A Stochastic Approximation Method"**
- 📄 [Link al paper](https://projecteuclid.org/journals/annals-of-mathematical-statistics/volume-22/issue-3/A-Stochastic-Approximation-Method/10.1214/aoms/1177729586.full)
- 🏆 **Paper fundamental** que introdujo el método de aproximación estocástica
- 📊 **3,700+ citas**
- 💡 **Por qué leerlo:** Base matemática del SGD, demuestra convergencia bajo condiciones específicas
- ⭐ **Impacto:** Fundación teórica de todos los optimizadores modernos

#### 2. **Cauchy (1847)** - Método del Descenso Más Pronunciado
**"Méthode générale pour la résolution des systèmes d'équations simultanées"**
- 📄 Primer método de gradient descent documentado
- 💡 **Contexto histórico:** Propuesto para resolver sistemas de ecuaciones
- ⭐ **Legado:** Base de todos los métodos de optimización de primer orden

---

### 🚀 Papers de Optimizadores Clásicos

#### 3. **Polyak (1964)** - Momentum
**"Some methods of speeding up the convergence of iteration methods"**
- 📄 [Link al paper](https://www.researchgate.net/publication/243648538_Some_methods_of_speeding_up_the_convergence_of_iteration_methods)
- 🏆 Introdujo el concepto de **momentum** en optimización
- 📊 **8,000+ citas**
- 💡 **Idea clave:** Acumular velocidad en direcciones consistentes para acelerar convergencia
- 📝 **Fórmula:** $v_t = \beta v_{t-1} + \nabla f$, $x_{t+1} = x_t - \alpha v_t$

#### 4. **Sutskever et al. (2013)** - Momentum en Deep Learning
**"On the importance of initialization and momentum in deep learning"**
- 📄 [Link al paper](http://proceedings.mlr.press/v28/sutskever13.pdf)
- 🏆 **Demostró la importancia crítica** de momentum para entrenar redes profundas
- 📊 **8,500+ citas**
- 💡 **Contribuciones:**
  - Nesterov Accelerated Gradient (NAG) para deep learning
  - Análisis de convergencia en funciones no convexas
  - Recomendaciones prácticas ($\beta = 0.9$ o $0.99$)
- 🔬 **Experimentos:** ImageNet, MNIST, Speech Recognition

#### 5. **Tieleman & Hinton (2012)** - RMSprop
**"Lecture 6.5 - RMSprop: Divide the gradient by a running average of its recent magnitude"**
- 📄 [Link (Coursera Neural Networks)](https://www.cs.toronto.edu/~tijmen/csc321/slides/lecture_slides_lec6.pdf)
- 🏆 Propuesto en el curso de Hinton, no paper formal
- 💡 **Idea:** Adaptar learning rate por parámetro usando promedio móvil de gradientes al cuadrado
- 📝 **Fórmula:** $s_t = 0.9 s_{t-1} + 0.1 g_t^2$, $\theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{s_t + \epsilon}} g_t$
- ⭐ **Uso:** Especialmente efectivo para RNNs

#### 6. **Kingma & Ba (2014)** - Adam
**"Adam: A Method for Stochastic Optimization"**
- 📄 [arXiv:1412.6980](https://arxiv.org/abs/1412.6980)
- 🏆 **El paper más influyente** en optimización moderna
- 📊 **90,000+ citas** (uno de los más citados en ML)
- 💡 **Contribuciones:**
  - Combina Momentum + RMSprop
  - Corrección de sesgo para momentos
  - Hiperparámetros por defecto que funcionan bien
- 📝 **Hiperparámetros recomendados:** $\alpha = 0.001$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$
- ⭐ **Por qué domina:** "Funciona out of the box" en la mayoría de problemas
- 🔬 **Experimentos:** Logistic regression, MLP, CNNs

---

### 🔥 Papers de Variantes de Adam

#### 7. **Loshchilov & Hutter (2017)** - AdamW
**"Decoupled Weight Decay Regularization"**
- 📄 [arXiv:1711.05101](https://arxiv.org/abs/1711.05101)
- 🏆 **Corrige un bug fundamental en Adam**
- 📊 **4,500+ citas**
- 💡 **Problema identificado:** Adam no implementa correctamente L2 regularization
- 📝 **Solución:** Desacoplar weight decay de la actualización del gradiente
- ⭐ **Resultado:** Mejor generalización que Adam en muchos casos
- 🔬 **Uso:** Optimizador por defecto en PyTorch para Transformers

#### 8. **Chen et al. (2023)** - Lion
**"Symbolic Discovery of Optimization Algorithms"**
- 📄 [arXiv:2302.06675](https://arxiv.org/abs/2302.06675)
- 🏆 **Descubierto por búsqueda algorítmica automática**
- 💡 **Idea radical:** Usar el signo del momentum en lugar del valor
- 📝 **Ventajas:**
  - Más eficiente en memoria que Adam
  - Mejor o igual performance
  - Solo un hiperparámetro (vs 2 en Adam)
- ⭐ **Uso:** Google lo usa para entrenar modelos grandes (PaLM)
- 🔬 **Experimentos:** ImageNet, ViT, Transformers

#### 9. **Liu et al. (2023)** - Sophia
**"Sophia: A Scalable Stochastic Second-order Optimizer for Language Model Pre-training"**
- 📄 [arXiv:2305.14342](https://arxiv.org/abs/2305.14342)
- 🏆 **Optimizador de segundo orden escalable**
- 💡 **Idea:** Usar información de segunda derivada (curvatura) sin el costo prohibitivo
- 📝 **Ventaja:** 2x más rápido que Adam para entrenar LLMs
- ⭐ **Uso:** Pre-entrenamiento de modelos de lenguaje grandes
- 🔬 **Experimentos:** GPT-2, GPT-Medium (hasta 770M parámetros)

---

### 📈 Papers sobre Learning Rate Schedules

#### 10. **Smith (2017)** - Cyclical Learning Rates
**"Cyclical Learning Rates for Training Neural Networks"**
- 📄 [arXiv:1506.01186](https://arxiv.org/abs/1506.01186)
- 🏆 Introduce **learning rate cíclico** en lugar de decreciente
- 📊 **2,100+ citas**
- 💡 **Idea:** Variar learning rate entre bounds durante entrenamiento
- 📝 **Método:** Triangular, triangular2, exp_range
- ⭐ **Beneficio:** Puede escapar mínimos locales, converge más rápido
- 🔬 **Herramienta:** LR Range Test para encontrar learning rate óptimo

#### 11. **Loshchilov & Hutter (2016)** - SGDR
**"SGDR: Stochastic Gradient Descent with Warm Restarts"**
- 📄 [arXiv:1608.03983](https://arxiv.org/abs/1608.03983)
- 🏆 **Cosine annealing con restarts**
- 📊 **2,800+ citas**
- 💡 **Idea:** Reducir lr con cosine annealing, luego hacer "warm restart"
- 📝 **Fórmula:** $\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})(1 + \cos(\frac{T_{cur}}{T_i}\pi))$
- ⭐ **Uso:** Muy popular en computer vision (ResNets, etc.)

---

### 🎓 Papers sobre Teoría de Convergencia

#### 12. **Bottou (2010)** - Large-Scale Machine Learning with SGD
**"Large-Scale Machine Learning with Stochastic Gradient Descent"**
- 📄 [Link al paper](http://leon.bottou.org/publications/pdf/compstat-2010.pdf)
- 🏆 **Análisis teórico profundo de SGD**
- 📊 **3,200+ citas**
- 💡 **Contribuciones:**
  - Trade-off entre exactitud y complejidad computacional
  - Por qué SGD domina en ML a gran escala
  - Análisis de convergencia vs Batch GD
- ⭐ **Conclusión:** Para datasets grandes, SGD es óptimo

#### 13. **Reddi et al. (2018)** - Convergence of Adam
**"On the Convergence of Adam and Beyond"**
- 📄 [arXiv:1904.09237](https://arxiv.org/abs/1904.09237)
- 🏆 **Identifica problema de convergencia en Adam**
- 📊 **1,700+ citas**
- 💡 **Problema:** Adam puede no converger en ciertos casos
- 📝 **Solución:** Proponen AMSGrad (usa max de v en lugar de promedio)
- ⭐ **Impacto:** Generó debate sobre robustez de Adam

---

### 🛠️ Papers sobre Aspectos Prácticos

#### 14. **Goyal et al. (2017)** - Scaling Learning Rate
**"Accurate, Large Minibatch SGD: Training ImageNet in 1 Hour"**
- 📄 [arXiv:1706.02677](https://arxiv.org/abs/1706.02677)
- 🏆 **Linear Scaling Rule** para batch size
- 📊 **2,100+ citas**
- 💡 **Regla:** Cuando multiplicas batch size por k, multiplica learning rate por k
- 📝 **Fórmula:** $lr_{new} = lr_{base} \times \frac{batch\_size_{new}}{batch\_size_{base}}$
- ⭐ **Aplicación:** Entrenar ResNet-50 en ImageNet en 1 hora (batch size 8192)
- 🔬 **Técnicas adicionales:** Warmup, learning rate scheduling

#### 15. **Smith & Topin (2017)** - Super-Convergence
**"Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates"**
- 📄 [arXiv:1708.07120](https://arxiv.org/abs/1708.07120)
- 🏆 **Entrenar redes en 10x menos epochs**
- 📊 **800+ citas**
- 💡 **Idea:** Usar learning rates muy grandes (2-10x lo normal) con 1cycle policy
- 📝 **1cycle policy:**
  1. Aumentar lr de base a max (50% de epochs)
  2. Disminuir lr de max a base (50% de epochs)
  3. Annealing final a casi cero
- ⭐ **Resultado:** CIFAR-10 en 10 epochs (vs 100 tradicional)
- 🔬 **Uso:** Implementado en fast.ai library

---

### 📖 Libros de Referencia

#### 16. **Goodfellow, Bengio & Courville (2016)** - Deep Learning
- 📘 [Link al libro (gratuito)](https://www.deeplearningbook.org/)
- **Capítulo 8: Optimization for Training Deep Models**
- 💡 **Contenido:**
  - Challenges en optimización de redes profundas
  - Análisis detallado de todos los optimizadores
  - Ill-conditioning, gradientes explosivos/vanishing
  - Batch normalization y su efecto en optimización
- ⭐ **Por qué leerlo:** Tratamiento comprehensivo y riguroso

#### 17. **Bottou, Curtis & Nocedal (2018)** - Optimization Methods
**"Optimization Methods for Large-Scale Machine Learning"**
- 📄 [arXiv:1606.04838](https://arxiv.org/abs/1606.04838)
- 📘 **Survey completo de 100+ páginas**
- 📊 **2,000+ citas**
- 💡 **Cubre:**
  - Teoría de convergencia rigurosa
  - Métodos de primer y segundo orden
  - Análisis de varianza en SGD
  - Paralelización y distribución
- ⭐ **Nivel:** Avanzado, con matemáticas rigurosas

---

### 🌐 Recursos Interactivos y Tutoriales

#### 18. **Distill.pub - "Why Momentum Really Works"**
- 🌐 [https://distill.pub/2017/momentum/](https://distill.pub/2017/momentum/)
- 💡 **Visualizaciones interactivas excelentes**
- ⭐ **Explica:** Intuición geométrica de momentum
- 🎯 **Nivel:** Accesible para beginners

#### 19. **CS231n - Optimization Notes**
- 🌐 [http://cs231n.github.io/optimization-1/](http://cs231n.github.io/optimization-1/)
- 💡 **Stanford course notes** (Andrej Karpathy)
- ⭐ **Contenido:**
  - Visualizaciones de loss landscapes
  - Comparación práctica de optimizadores
  - Tips y tricks

#### 20. **Ruder (2016)** - Overview of Gradient Descent
- 📄 [arXiv:1609.04747](https://arxiv.org/abs/1609.04747)
- 💡 **Blog post comprehensivo convertido en paper**
- 📊 **8,500+ citas**
- ⭐ **Cubre:** Todos los optimizadores principales con intuiciones claras
- 🎯 **Perfecto para:** Overview rápido de todos los métodos
- 🌐 [Blog post original](https://ruder.io/optimizing-gradient-descent/)

---

### 📊 Tabla Resumen de Papers Clave

| Paper | Año | Contribución | Citas | Por qué leerlo |
|-------|-----|--------------|-------|----------------|
| Robbins & Monro | 1951 | Fundamentos de SGD | 3,700+ | Base teórica |
| Polyak | 1964 | Momentum | 8,000+ | Aceleración |
| Sutskever et al. | 2013 | Momentum en DL | 8,500+ | Práctica moderna |
| Kingma & Ba | 2014 | Adam | 90,000+ | **Más usado hoy** |
| Loshchilov & Hutter | 2017 | AdamW | 4,500+ | Corrección de Adam |
| Chen et al. | 2023 | Lion | Nuevo | **Estado del arte** |
| Liu et al. | 2023 | Sophia | Nuevo | LLMs |
| Smith | 2017 | Cyclical LR | 2,100+ | Práctico |
| Goyal et al. | 2017 | Scaling rule | 2,100+ | Batch size grande |
| Bottou et al. | 2018 | Survey | 2,000+ | **Lectura obligatoria** |

---

### 🎯 Guía de Lectura Recomendada

**Para principiantes:**
1. 📄 Ruder (2016) - Overview
2. 🌐 Distill.pub - Momentum
3. 🌐 CS231n notes
4. 📄 Kingma & Ba (2014) - Adam paper

**Para nivel intermedio:**
5. 📄 Sutskever et al. (2013) - Momentum
6. 📄 Loshchilov & Hutter (2017) - AdamW
7. 📄 Smith (2017) - Cyclical LR
8. 📄 Goyal et al. (2017) - Scaling
9. 📘 Goodfellow et al. - Cap. 8

**Para investigadores:**
10. 📄 Robbins & Monro (1951) - Fundamentos
11. 📄 Bottou et al. (2018) - Survey completo
12. 📄 Reddi et al. (2018) - Convergencia
13. 📄 Chen et al. (2023) - Lion
14. 📄 Liu et al. (2023) - Sophia

---

### 💡 Preguntas para Profundizar

Después de leer estos papers, considera:

1. **¿Por qué Adam domina en práctica pero SGD+Momentum a veces generaliza mejor?**
   - Ver: Wilson et al. (2017) "The Marginal Value of Adaptive Gradient Methods"

2. **¿Cómo afecta la arquitectura del modelo al optimizador óptimo?**
   - CNNs: SGD+Momentum funciona muy bien
   - Transformers: AdamW es estándar
   - RNNs: RMSprop o Adam

3. **¿Batch size afecta la convergencia? ¿Y la generalización?**
   - Ver: Keskar et al. (2016) "On Large-Batch Training for Deep Learning"

4. **¿Hay un "optimizador universal" o siempre depende del problema?**
   - Spoiler: Lion y Sophia son intentos recientes de universalidad

---

[⬆ Volver a Tabla de Contenidos](#toc)

<a name='6'></a>

---
## 🏭 6. Comparación con Frameworks Modernos

Ahora que has implementado gradient descent desde cero, veamos cómo los frameworks modernos lo implementan y cuándo usar cada uno.

### 📊 Comparación de Performance: Manual vs Framework

Primero, comparemos el rendimiento de nuestra implementación vs frameworks profesionales.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

# Generar dataset grande para comparación
np.random.seed(42)
n_samples = 10000
n_features = 100

X_large = np.random.randn(n_samples, n_features)
true_weights = np.random.randn(n_features) * 0.5
y_large = X_large @ true_weights + np.random.randn(n_samples) * 0.1

print(f"📊 Dataset generado: {X_large.shape}")
print(f"   Features: {n_features}")
print(f"   Samples: {n_samples}")

In [ ]:
# Benchmark 1: Nuestra implementación manual
print("\n" + "="*60)
print("🔧 Benchmark 1: Implementación Manual (NumPy)")
print("="*60)

start_time = time.time()

model_manual = GradientDescentOptimizer(
    learning_rate=0.01,
    n_iterations=100,
    batch_size=32,
    optimizer='adam'
)
model_manual.fit(X_large, y_large)

manual_time = time.time() - start_time
manual_final_loss = model_manual.losses[-1]

print(f"\n⏱️  Tiempo total: {manual_time:.2f} segundos")
print(f"📉 Loss final: {manual_final_loss:.6f}")

### 🔥 PyTorch Implementation

PyTorch es uno de los frameworks más populares para deep learning. Veamos cómo implementar lo mismo.

In [ ]:
# Verificar si PyTorch está disponible
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    
    PYTORCH_AVAILABLE = True
    print("✅ PyTorch disponible")
    print(f"   Versión: {torch.__version__}")
    
    # Check for GPU
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device('cpu')
        print(f"   Device: CPU")
        
except ImportError:
    PYTORCH_AVAILABLE = False
    print("⚠️ PyTorch no está instalado")
    print("   Instalar con: pip install torch")

In [ ]:
if PYTORCH_AVAILABLE:
    print("\n" + "="*60)
    print("🔥 Benchmark 2: PyTorch")
    print("="*60)
    
    # Convertir a tensores de PyTorch
    X_torch = torch.FloatTensor(X_large).to(device)
    y_torch = torch.FloatTensor(y_large).to(device)
    
    # Crear modelo simple
    class LinearModel(nn.Module):
        def __init__(self, input_dim):
            super(LinearModel, self).__init__()
            self.linear = nn.Linear(input_dim, 1, bias=True)
            
        def forward(self, x):
            return self.linear(x).squeeze()
    
    # Crear DataLoader
    dataset = TensorDataset(X_torch, y_torch)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    # Inicializar modelo
    model_torch = LinearModel(n_features).to(device)
    optimizer_torch = optim.Adam(model_torch.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    
    # Training loop
    start_time = time.time()
    
    torch_losses = []
    model_torch.train()
    
    for epoch in range(100):
        epoch_loss = 0
        for X_batch, y_batch in dataloader:
            optimizer_torch.zero_grad()
            outputs = model_torch(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer_torch.step()
            epoch_loss += loss.item()
        
        torch_losses.append(epoch_loss / len(dataloader))
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d} - Loss: {torch_losses[-1]:.6f}")
    
    pytorch_time = time.time() - start_time
    pytorch_final_loss = torch_losses[-1]
    
    print(f"\n⏱️  Tiempo total: {pytorch_time:.2f} segundos")
    print(f"📉 Loss final: {pytorch_final_loss:.6f}")
    print(f"🚀 Speedup: {manual_time/pytorch_time:.2f}x más rápido que implementación manual")
else:
    print("\n⚠️ Saltando benchmark de PyTorch (no instalado)")

### 🧠 TensorFlow/Keras Implementation

TensorFlow/Keras es otro framework muy popular, especialmente en producción.

In [ ]:
# Verificar si TensorFlow está disponible
try:
    import tensorflow as tf
    from tensorflow import keras
    
    # Silenciar warnings
    tf.get_logger().setLevel('ERROR')
    
    TENSORFLOW_AVAILABLE = True
    print("✅ TensorFlow disponible")
    print(f"   Versión: {tf.__version__}")
    
    # Check for GPU
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"   GPUs disponibles: {len(gpus)}")
    else:
        print(f"   Device: CPU")
        
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("⚠️ TensorFlow no está instalado")
    print("   Instalar con: pip install tensorflow")

In [ ]:
if TENSORFLOW_AVAILABLE:
    print("\n" + "="*60)
    print("🧠 Benchmark 3: TensorFlow/Keras")
    print("="*60)
    
    # Crear modelo
    model_tf = keras.Sequential([
        keras.layers.Dense(1, input_shape=(n_features,), use_bias=True)
    ])
    
    # Compilar con Adam
    model_tf.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.01),
        loss='mse'
    )
    
    # Entrenar
    start_time = time.time()
    
    history = model_tf.fit(
        X_large, y_large,
        epochs=100,
        batch_size=32,
        verbose=0
    )
    
    tensorflow_time = time.time() - start_time
    tensorflow_final_loss = history.history['loss'][-1]
    
    print(f"\n⏱️  Tiempo total: {tensorflow_time:.2f} segundos")
    print(f"📉 Loss final: {tensorflow_final_loss:.6f}")
    print(f"🚀 Speedup: {manual_time/tensorflow_time:.2f}x más rápido que implementación manual")
else:
    print("\n⚠️ Saltando benchmark de TensorFlow (no instalado)")

### 📊 Visualización Comparativa de Convergencia

In [ ]:
# Crear visualización comparativa
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: Convergencia
ax1.plot(model_manual.losses, label='NumPy (Manual)', linewidth=2, alpha=0.8)
if PYTORCH_AVAILABLE:
    ax1.plot(torch_losses, label='PyTorch', linewidth=2, alpha=0.8)
if TENSORFLOW_AVAILABLE:
    ax1.plot(history.history['loss'], label='TensorFlow', linewidth=2, alpha=0.8)

ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Loss (MSE)', fontsize=12)
ax1.set_title('Convergencia: Manual vs Frameworks', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# Subplot 2: Tiempo de ejecución
times = [manual_time]
labels = ['NumPy\n(Manual)']
colors = ['#3498db']

if PYTORCH_AVAILABLE:
    times.append(pytorch_time)
    labels.append('PyTorch')
    colors.append('#e74c3c')
    
if TENSORFLOW_AVAILABLE:
    times.append(tensorflow_time)
    labels.append('TensorFlow')
    colors.append('#2ecc71')

bars = ax2.bar(labels, times, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_ylabel('Tiempo (segundos)', fontsize=12)
ax2.set_title('Tiempo de Entrenamiento (100 epochs)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Añadir valores en las barras
for bar, time_val in zip(bars, times):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{time_val:.2f}s',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("📊 RESUMEN DE BENCHMARKS")
print("="*60)
print(f"{'Framework':<20} {'Tiempo (s)':<15} {'Loss Final':<15} {'Speedup'}")
print("-"*60)
print(f"{'NumPy (Manual)':<20} {manual_time:<15.2f} {manual_final_loss:<15.6f} {'1.00x'}")
if PYTORCH_AVAILABLE:
    print(f"{'PyTorch':<20} {pytorch_time:<15.2f} {pytorch_final_loss:<15.6f} {manual_time/pytorch_time:.2f}x")
if TENSORFLOW_AVAILABLE:
    print(f"{'TensorFlow':<20} {tensorflow_time:<15.2f} {tensorflow_final_loss:<15.6f} {manual_time/tensorflow_time:.2f}x")
print("="*60)

### 🎯 Comparación de Optimizadores en PyTorch

<a name='7'></a>

---
## 🔬 7. Ejercicios Avanzados (Opcionales)

Estos ejercicios son **opcionales** y no cuentan para la calificación oficial. Son para estudiantes que quieren profundizar y explorar aspectos avanzados de gradient descent.

**💡 Beneficios:**
- Preparan para investigación en optimización
- Útiles para entrevistas técnicas avanzadas
- Te dan experiencia con técnicas de producción

---

### 🟣 Ejercicio Avanzado 1: Nesterov Accelerated Gradient

**Objetivo:** Implementa NAG, una variante de momentum que "mira hacia adelante"

#### Teoría

Nesterov Accelerated Gradient (NAG) mejora momentum calculando el gradiente en la posición anticipada:

**Momentum estándar:**
$$v_t = \beta v_{t-1} + \nabla J(\theta_{t-1})$$
$$\theta_t = \theta_{t-1} - \alpha v_t$$

**Nesterov Momentum:**
$$\theta_{lookahead} = \theta_{t-1} - \beta v_{t-1}$$
$$v_t = \beta v_{t-1} + \nabla J(\theta_{lookahead})$$
$$\theta_t = \theta_{t-1} - \alpha v_t$$

**Ventaja:** NAG "corrige" el momentum antes de aplicarlo, evitando overshooting.

#### Implementación

In [ ]:
def nesterov_momentum(w, b, X, y, v_w, v_b, learning_rate, beta=0.9):
    """
    Implementa Nesterov Accelerated Gradient.
    
    Arguments:
    w -- pesos actuales
    b -- bias actual
    X -- features
    y -- targets
    v_w -- velocidad de w
    v_b -- velocidad de b
    learning_rate -- tasa de aprendizaje
    beta -- coeficiente de momentum
    
    Returns:
    w_new, b_new, v_w_new, v_b_new
    """
    
    # PASO 1: Calcular posición lookahead
    # TODO: Implementa w_lookahead = w - beta * v_w
    w_lookahead = None  # TU CÓDIGO AQUÍ
    b_lookahead = None  # TU CÓDIGO AQUÍ
    
    # PASO 2: Calcular gradiente en posición lookahead
    # TODO: Usa compute_gradient() con parámetros lookahead
    dw, db = None, None  # TU CÓDIGO AQUÍ
    
    # PASO 3: Actualizar velocidad
    # TODO: v = beta * v + gradient
    v_w_new = None  # TU CÓDIGO AQUÍ
    v_b_new = None  # TU CÓDIGO AQUÍ
    
    # PASO 4: Actualizar parámetros
    # TODO: theta = theta - lr * v
    w_new = None  # TU CÓDIGO AQUÍ
    b_new = None  # TU CÓDIGO AQUÍ
    
    return w_new, b_new, v_w_new, v_b_new

# Test básico (descomenta para probar)
# w_test = np.array([1.0, 2.0])
# b_test = 0.5
# v_w_test = np.array([0.1, 0.2])
# v_b_test = 0.05
# X_test = np.random.randn(100, 2)
# y_test = np.random.randn(100)
# w_new, b_new, v_w_new, v_b_new = nesterov_momentum(w_test, b_test, X_test, y_test, v_w_test, v_b_test, 0.01)
# print(f"Nesterov update: w={w_new}, b={b_new}")

#### Comparación: Momentum vs Nesterov

Compara la convergencia de momentum estándar vs Nesterov en un problema difícil.

In [ ]:
# Comparación visual Momentum vs Nesterov
# TODO: Implementa un entrenamiento completo con ambos métodos
# Sugerencia: Usa un problema con alta curvatura (conditioning number alto)

def compare_momentum_vs_nesterov():
    """
    Compara convergencia de momentum estándar vs Nesterov.
    """
    # Generar datos con alta correlación (problema difícil)
    # TODO: Tu implementación aquí
    pass

# compare_momentum_vs_nesterov()

---

### 🟣 Ejercicio Avanzado 2: Visualización de Loss Landscapes

**Objetivo:** Visualiza cómo diferentes optimizadores navegan el espacio de parámetros

#### Implementación

In [ ]:
def visualize_optimization_paths(X, y, optimizers, n_iterations=100):
    """
    Visualiza las trayectorias de diferentes optimizadores en el espacio de parámetros.
    
    Arguments:
    X -- features (debe tener 1 o 2 features para visualización)
    y -- targets
    optimizers -- dict de optimizadores a comparar
    n_iterations -- número de iteraciones
    """
    
    # PASO 1: Entrenar cada optimizador y guardar historial de parámetros
    paths = {}
    
    for opt_name, opt_config in optimizers.items():
        print(f"Entrenando {opt_name}...")
        
        # TODO: Entrena el modelo y guarda weight_history
        # Sugerencia: Modifica GradientDescentOptimizer para guardar todos los pasos
        pass
    
    # PASO 2: Crear grid para superficie de loss
    if X.shape[1] <= 2:
        # TODO: Crea meshgrid de w y b
        # TODO: Calcula MSE para cada punto
        # TODO: Crea visualización 3D con Plotly
        pass
    else:
        print("⚠️ Visualización solo disponible para 1-2 features")
    
    return paths

# Ejemplo de uso:
# optimizers_to_compare = {
#     'SGD': {'optimizer': 'gd', 'learning_rate': 0.01},
#     'Momentum': {'optimizer': 'momentum', 'learning_rate': 0.01},
#     'RMSprop': {'optimizer': 'rmsprop', 'learning_rate': 0.01},
#     'Adam': {'optimizer': 'adam', 'learning_rate': 0.01}
# }
# paths = visualize_optimization_paths(X_train[:1000], y_train[:1000], optimizers_to_compare)

---

### 🟣 Ejercicio Avanzado 3: Learning Rate Finder

**Objetivo:** Implementa el LR Range Test de Leslie Smith

#### Teoría

El LR Range Test encuentra el learning rate óptimo:
1. Empieza con lr muy pequeño (1e-8)
2. Aumenta exponencialmente cada batch
3. Grafica loss vs learning rate
4. El lr óptimo está donde el loss disminuye más rápido (mayor pendiente negativa)

**Referencia:** Smith (2017) - "Cyclical Learning Rates for Training Neural Networks"

#### Implementación

In [ ]:
def lr_range_test(X, y, start_lr=1e-8, end_lr=10, num_iterations=100):
    """
    Implementa Learning Rate Range Test.
    
    Arguments:
    X -- features
    y -- targets
    start_lr -- learning rate inicial (muy pequeño)
    end_lr -- learning rate final (muy grande)
    num_iterations -- número de iteraciones
    
    Returns:
    lrs -- array de learning rates probados
    losses -- array de losses correspondientes
    suggested_lr -- learning rate sugerido
    """
    
    # Inicializar
    n_features = X.shape[1]
    w = np.zeros(n_features)
    b = 0.0
    
    # Arrays para guardar resultados
    lrs = []
    losses = []
    
    # Factor multiplicativo para aumentar lr exponencialmente
    mult = (end_lr / start_lr) ** (1 / num_iterations)
    lr = start_lr
    
    # Mini-batches
    batch_size = 32
    
    print("🔍 Buscando learning rate óptimo...")
    print(f"   Rango: {start_lr:.2e} a {end_lr:.2e}")
    print(f"   Iteraciones: {num_iterations}\n")
    
    for iteration in range(num_iterations):
        # TODO: PASO 1 - Seleccionar mini-batch aleatorio
        
        # TODO: PASO 2 - Calcular gradiente
        
        # TODO: PASO 3 - Actualizar parámetros con lr actual
        
        # TODO: PASO 4 - Calcular loss en TODO el dataset
        
        # Guardar resultados
        lrs.append(lr)
        # losses.append(current_loss)  # TODO: añadir loss calculado
        
        # Aumentar learning rate
        lr *= mult
        
        # Stop si loss explota
        # if current_loss > losses[0] * 4:  # TODO: descomentar
        #     break
        
        if iteration % 20 == 0:
            pass  # print(f"Iteration {iteration}: lr={lr:.2e}")  # TODO
    
    # TODO: PASO 5 - Encontrar lr óptimo (mayor pendiente negativa)
    # Sugerencia: calcula gradiente numérico de losses y encuentra mínimo
    suggested_lr = None  # TODO
    
    # Visualización
    # TODO: Grafica loss vs learning rate (escala log en x)
    
    return lrs, losses, suggested_lr

# Ejemplo de uso:
# lrs, losses, suggested_lr = lr_range_test(X_train, y_train)
# print(f"\n✅ Learning rate sugerido: {suggested_lr:.2e}")

---

### 🟣 Ejercicio Avanzado 4: Gradient Clipping

**Objetivo:** Implementa gradient clipping para prevenir gradientes explosivos

#### Teoría

Gradient clipping es crítico para entrenar RNNs y Transformers:

**1. Clipping by value:**
$$g_{clipped} = \max(\min(g, threshold), -threshold)$$

**2. Clipping by norm:**
$$g_{clipped} = \begin{cases}
g & \text{if } \|g\| \leq threshold \\
\frac{threshold}{\|g\|} \cdot g & \text{otherwise}
\end{cases}$$

#### Implementación

In [ ]:
def clip_gradient_by_norm(gradient, max_norm=1.0):
    """
    Clip gradient by its L2 norm.
    
    Arguments:
    gradient -- gradiente a clip (array)
    max_norm -- norma máxima permitida
    
    Returns:
    gradient_clipped -- gradiente clippeado
    """
    # TODO: Calcula la norma L2 del gradiente
    norm = None  # TU CÓDIGO AQUÍ (usa np.linalg.norm)
    
    # TODO: Si norm > max_norm, escala el gradiente
    if norm > max_norm:
        gradient_clipped = None  # TU CÓDIGO AQUÍ
    else:
        gradient_clipped = gradient
    
    return gradient_clipped

def clip_gradient_by_value(gradient, clip_value=1.0):
    """
    Clip gradient by value.
    
    Arguments:
    gradient -- gradiente a clip
    clip_value -- valor máximo absoluto
    
    Returns:
    gradient_clipped
    """
    # TODO: Usa np.clip para limitar valores
    gradient_clipped = None  # TU CÓDIGO AQUÍ
    
    return gradient_clipped

# Tests
print("Testing gradient clipping...")

# Test 1: Gradient pequeño (no debe cambiar)
g_small = np.array([0.1, 0.2, 0.3])
print(f"\nGradiente pequeño: {g_small}")
# g_clipped = clip_gradient_by_norm(g_small, max_norm=1.0)
# print(f"Después de clip: {g_clipped}")

# Test 2: Gradient grande (debe clippear)
g_large = np.array([10.0, 20.0, 30.0])
print(f"\nGradiente grande: {g_large}")
# g_clipped = clip_gradient_by_norm(g_large, max_norm=1.0)
# print(f"Después de clip: {g_clipped}")
# print(f"Norma después de clip: {np.linalg.norm(g_clipped):.4f} (debe ser ≤ 1.0)")

---

### 🟣 Ejercicio Avanzado 5: Análisis de Convergencia Estocástica

**Objetivo:** Estudia el comportamiento estocástico de SGD

#### Experimento

In [ ]:
def analyze_stochastic_convergence(X, y, n_runs=10, batch_size=32):
    """
    Analiza la varianza en convergencia de SGD.
    
    Arguments:
    X, y -- datos
    n_runs -- número de entrenamientos con diferentes seeds
    batch_size -- tamaño de batch para SGD
    """
    
    all_losses = []
    final_params = []
    
    print(f"🔬 Analizando convergencia estocástica con {n_runs} runs...\n")
    
    for run in range(n_runs):
        # TODO: Entrena modelo con seed diferente
        # np.random.seed(run)
        # model = ...
        # all_losses.append(model.losses)
        # final_params.append((model.weights, model.bias))
        pass
    
    # Convertir a arrays
    # all_losses = np.array(all_losses)  # shape: (n_runs, n_iterations)
    
    # TODO: Calcular estadísticas
    # mean_loss = np.mean(all_losses, axis=0)
    # std_loss = np.std(all_losses, axis=0)
    
    # TODO: Visualizar
    # fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Subplot 1: Todas las curvas
    # for i, losses in enumerate(all_losses):
    #     ax1.plot(losses, alpha=0.3, color='blue')
    # ax1.plot(mean_loss, color='red', linewidth=2, label='Media')
    # ax1.fill_between(range(len(mean_loss)), 
    #                  mean_loss - std_loss, 
    #                  mean_loss + std_loss,
    #                  alpha=0.2, color='red', label='±1 std')
    
    # Subplot 2: Distribución de parámetros finales
    # TODO: Histograma de w y b finales
    
    print("\n📊 Análisis completado")
    # print(f"   Varianza del loss final: {std_loss[-1]:.6f}")
    
    return all_losses, final_params

# Ejemplo de uso:
# all_losses, final_params = analyze_stochastic_convergence(X_train, y_train, n_runs=10)

---

### 📚 Recursos para Ejercicios Avanzados

**Papers:**
- Sutskever et al. (2013) - Nesterov Momentum
- Smith (2017) - LR Finder y Cyclical LR
- Pascanu et al. (2013) - "On the difficulty of training RNNs" (gradient clipping)
- Goodfellow et al. (2016) - Deep Learning Book, Cap. 8

**Código de Referencia:**
- PyTorch: `torch.optim.SGD(nesterov=True)`
- PyTorch: `torch.nn.utils.clip_grad_norm_()`
- Fast.ai: `lr_find()` method

**Herramientas:**
- TensorBoard para visualizar loss landscapes
- Weights & Biases para tracking de experimentos

---

[⬆ Volver a Tabla de Contenidos](#toc)

<a name='9'></a>

---
## 📚 9. Resumen y Mejores Prácticas

### 🎯 Conceptos Clave que Dominaste

**1. Gradient Descent - El Algoritmo Fundamental**
- ✅ Regla de actualización: $\theta := \theta - \alpha \nabla J(\theta)$
- ✅ Funciona para cualquier función diferenciable
- ✅ Base de prácticamente todos los modelos de ML/DL

**2. Las Tres Variantes**
- ✅ **Batch GD:** Usa todos los datos → estable pero lento
- ✅ **SGD:** Usa un ejemplo → rápido pero ruidoso
- ✅ **Mini-batch:** Balance óptimo → más usado en práctica

**3. Optimizadores Avanzados Implementados**
- ✅ **Momentum** (β=0.9): Acelera en direcciones consistentes
- ✅ **RMSprop** (β₂=0.999): Adapta LR por parámetro
- ✅ **Adam** (β₁=0.9, β₂=0.999): Combina ambos + bias correction
- ✅ **Learning Rate Decay**: Refina convergencia

**4. Papers Fundamentales Estudiados**
- ✅ 20+ papers curados desde Robbins & Monro (1951) hasta Sophia (2023)
- ✅ Adam paper: 90,000+ citas - el más influyente
- ✅ Papers recientes: Lion, AdamW, Sophia

---

### 🛠️ Guía de Decisión: ¿Qué Optimizador Usar?

Esta es una guía práctica basada en años de investigación y experiencia en producción.

#### Por Tipo de Problema

| Problema | Optimizador Recomendado | Hiperparámetros | Razón |
|----------|------------------------|-----------------|-------|
| **Computer Vision (CNNs)** | SGD + Momentum | lr=0.1, β=0.9 | Mejor generalización |
| **NLP (Transformers)** | AdamW | lr=1e-4, wd=0.01 | Estándar en BERT, GPT |
| **RNNs/LSTMs** | RMSprop o Adam | lr=1e-3 | Maneja gradientes variables |
| **Reinforcement Learning** | Adam | lr=3e-4 | Usado en PPO, SAC |
| **GANs** | Adam | lr=2e-4, β₁=0.5 | Ayuda con inestabilidad |
| **Modelos Muy Grandes (LLMs)** | Lion o AdamW | lr=1e-5 | Eficiencia de memoria |
| **ML Clásico (pequeño)** | Batch GD o Adam | lr=0.01 | Simple y efectivo |
| **Exploración rápida** | Adam | lr=1e-3 | Funciona "out of box" |

#### Por Tamaño de Dataset

| Dataset | Batch Size | Optimizador | Learning Rate |
|---------|------------|-------------|---------------|
| < 1K samples | Full batch | Batch GD | 0.01 - 0.1 |
| 1K - 100K | 32-128 | Adam o SGD+M | 0.001 - 0.01 |
| 100K - 1M | 128-512 | Adam o AdamW | 0.0001 - 0.001 |
| > 1M (huge) | 512-8192 | AdamW o Lion | 0.00001 - 0.0001 |

**Linear Scaling Rule:** Si multiplicas batch size por k, multiplica LR por k
- Ejemplo: batch=64, lr=0.001 → batch=256, lr=0.004
- Referencia: Goyal et al. (2017)

---

### ⚡ Mejores Prácticas y Tips de Producción

#### 1. Inicialización del Learning Rate

**❌ No hagas:**
```python
# Usar el mismo LR para todo
lr = 0.01  # ¿Por qué 0.01?
```

**✅ Haz esto:**
```python
# Usa LR Finder (Smith 2017)
from lr_finder import LRFinder
lr_finder = LRFinder(model, optimizer, criterion)
lr_finder.range_test(train_loader)
suggested_lr = lr_finder.get_best_lr()

# O usa valores empíricos probados
lr_by_optimizer = {
    'sgd': 0.1,
    'adam': 0.001,
    'adamw': 0.0001
}
```

#### 2. Learning Rate Scheduling

**Siempre usa algún tipo de LR decay**

**Opciones populares:**
```python
# 1. Step Decay (simple, efectivo)
# Reduce LR por factor cada N epochs
lr = lr_initial * (decay_rate ** (epoch // step_size))

# 2. Cosine Annealing (muy popular en CV)
lr = lr_min + 0.5 * (lr_max - lr_min) * (1 + cos(pi * epoch / total_epochs))

# 3. Reduce on Plateau (adaptativo)
# Reduce LR cuando val loss deja de mejorar

# 4. 1Cycle Policy (super-convergence)
# Aumenta y luego disminuye LR - converge en 10x menos epochs
```

**Recomendación:**
- CNNs: Cosine Annealing con Warm Restarts
- Transformers: Linear warmup + Linear decay
- General: Reduce on Plateau (más seguro)

#### 3. Warmup

**Para modelos grandes, SIEMPRE usa warmup:**
```python
warmup_epochs = 5
total_epochs = 100

if epoch < warmup_epochs:
    # Aumenta linealmente de 0 a lr_target
    lr = lr_target * (epoch / warmup_epochs)
else:
    # Luego aplica tu schedule normal
    lr = lr_schedule(epoch - warmup_epochs)
```

**Por qué:** Adam/AdamW tienen sesgo al inicio (bias correction no es suficiente)

#### 4. Gradient Clipping

**Para RNNs, Transformers, GANs → OBLIGATORIO:**
```python
# PyTorch
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# TensorFlow
optimizer = tf.keras.optimizers.Adam(clipnorm=1.0)
```

**Valores típicos:** max_norm=0.5 (RNNs), max_norm=1.0 (Transformers)

#### 5. Batch Size

**Trade-offs:**
- **Pequeño (8-32):** Más ruidoso, mejor generalización, más iteraciones
- **Grande (256-8192):** Más estable, peor generalización, menos iteraciones

**Regla práctica:**
- Empieza con 32 o 64
- Si tienes GPU memory, aumenta a 128-256
- Para datasets muy grandes: 512-1024
- Usa largest batch that fits in memory

**Importante:** Si aumentas batch size, aumenta learning rate proporcionalmente

#### 6. Weight Decay (L2 Regularization)

**Para Adam/AdamW:**
```python
# ❌ Mal - Adam no implementa L2 correctamente
optimizer = Adam(lr=0.001, weight_decay=0.01)

# ✅ Bien - AdamW desacopla weight decay
optimizer = AdamW(lr=0.001, weight_decay=0.01)
```

**Valores típicos:** 0.01 (moderado), 0.0001 (ligero), 0.1 (fuerte)

#### 7. Monitoring y Debugging

**Qué monitorear:**
```python
# 1. Training loss (debe bajar)
# 2. Validation loss (debe bajar, luego estabilizar)
# 3. Learning rate (para verificar schedule)
# 4. Gradient norms (detectar explosión/vanishing)
# 5. Weight norms (detectar divergencia)
```

**Signos de problemas:**
- Loss = NaN → LR muy alto, gradientes explotan
- Loss no baja → LR muy bajo, mal inicialización
- Train loss baja, val loss sube → Overfitting (usa regularización)
- Loss oscila mucho → LR muy alto, reduce o usa Momentum

---

### 🎯 Casos de Uso Reales

#### Caso 1: Entrenar ResNet-50 en ImageNet

**Setup usado en papers:**
```python
optimizer = SGD(
    params=model.parameters(),
    lr=0.1,  # Base LR
    momentum=0.9,
    weight_decay=0.0001
)

# Learning rate schedule
# Divide por 10 en epochs 30, 60, 90 (de 90 total)
scheduler = MultiStepLR(optimizer, milestones=[30, 60], gamma=0.1)

# Batch size: 256 (o 512-8192 con Linear Scaling Rule)
# Total epochs: 90
```

**Por qué funciona:**
- SGD+Momentum generaliza mejor que Adam en CNNs
- Step decay simple pero efectivo
- Batch size grande aprovecha GPUs

#### Caso 2: Fine-tuning BERT

**Setup estándar:**
```python
optimizer = AdamW(
    params=model.parameters(),
    lr=2e-5,  # Muy bajo para fine-tuning
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01
)

# Warmup: 10% de total steps
# Linear decay después de warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0.1 * total_steps,
    num_training_steps=total_steps
)

# Batch size: 16-32 (por GPU constraints)
# Epochs: 2-4 (fine-tuning es rápido)
```

**Por qué funciona:**
- AdamW corrige weight decay de Adam
- LR muy bajo para no destruir pre-training
- Warmup estabiliza inicio

#### Caso 3: Entrenar GPT desde cero

**Setup de OpenAI:**
```python
optimizer = AdamW(
    params=model.parameters(),
    lr=6e-4,  # Máximo LR (se alcanza después de warmup)
    betas=(0.9, 0.95),  # β₂ más bajo que default
    weight_decay=0.1
)

# Warmup: 2000 steps
# Cosine decay a 10% del max LR
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=total_steps,
    eta_min=6e-5  # 10% de 6e-4
)

# Batch size: muy grande (1024+ tokens por batch)
# Gradient accumulation si no cabe en memoria
```

**Alternativa moderna:**
```python
# Google usa Lion para modelos grandes
optimizer = Lion(
    params=model.parameters(),
    lr=1e-4,  # 3-10x menor que AdamW
    betas=(0.9, 0.99)
)
```

---

### 🔬 Troubleshooting Guide

#### Problema 1: Loss no baja

**Posibles causas:**
1. **Learning rate muy bajo**
   - Solución: Aumenta LR 10x, usa LR Finder
2. **Mala inicialización de pesos**
   - Solución: Usa He initialization (ReLU) o Xavier (sigmoid/tanh)
3. **Gradiente vanishing**
   - Solución: Batch normalization, mejor arquitectura, ReLU
4. **Bug en código**
   - Solución: Overfit a 1 batch (debe llegar a loss≈0)

#### Problema 2: Loss = NaN

**Causas:**
1. **Learning rate muy alto**
   - Solución: Reduce LR 10x o 100x
2. **Gradientes explotan**
   - Solución: Gradient clipping, reduce LR
3. **División por cero**
   - Solución: Revisa loss function, añade epsilon

#### Problema 3: Train baja, Val sube (Overfitting)

**Soluciones (en orden):**
1. Aumenta weight decay (0.001 → 0.01 → 0.1)
2. Añade dropout
3. Reduce tamaño del modelo
4. Consigue más datos
5. Data augmentation
6. Early stopping

#### Problema 4: Convergencia muy lenta

**Soluciones:**
1. Aumenta learning rate
2. Usa Adam en lugar de SGD (más rápido)
3. Aumenta batch size
4. Usa Momentum (β=0.9)
5. Batch normalization
6. Mejor inicialización

---

### 📊 Hoja de Referencia Rápida

**Configuraciones por defecto recomendadas:**

```python
# Computer Vision (CNNs)
optimizer = SGD(lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler = CosineAnnealingLR(T_max=epochs)
batch_size = 128

# NLP (Transformers)
optimizer = AdamW(lr=1e-4, betas=(0.9, 0.999), weight_decay=0.01)
scheduler = LinearWarmupCosineDecay(warmup=0.1*steps)
batch_size = 32

# Exploración rápida
optimizer = Adam(lr=1e-3)
# No scheduler
batch_size = 32

# Modelos muy grandes (GPT, etc)
optimizer = Lion(lr=1e-4, betas=(0.9, 0.99)) # o AdamW
scheduler = CosineAnnealingLR(T_max=steps, eta_min=1e-5)
batch_size = 512+ (con gradient accumulation)
```

---

### 🚀 Próximos Pasos en tu Viaje de Optimización

**Has completado:**
- ✅ Fundamentos teóricos sólidos
- ✅ Implementación desde cero de 8 funciones
- ✅ 100 puntos de ejercicios autogradeados
- ✅ Estudio de 20+ papers fundamentales
- ✅ Ejercicios avanzados opcionales

**Ahora estás preparado para:**

1. **Siguiente notebook: 03. Regresión Logística**
   - Aplicar gradient descent a clasificación
   - Cross-entropy loss
   - Función sigmoide y probabilidades

2. **Deep Learning**
   - Entrenar redes neuronales
   - Backpropagation
   - Transfer learning

3. **Proyectos Reales**
   - Kaggle competitions
   - Fine-tuning LLMs
   - Entrenar tus propios modelos

4. **Investigación**
   - Leer papers avanzados
   - Implementar nuevos optimizadores
   - Contribuir a PyTorch/TensorFlow

---

### 🏆 ¡Felicitaciones!

Has dominado **el corazón del Machine Learning**. Gradient descent es la base sobre la que se construyen:

- 🤖 Todos los modelos de Deep Learning (CNNs, Transformers, GANs, Diffusion)
- 📊 ML clásico (regresión, SVM, boosting)
- 🎮 Reinforcement Learning (policy gradients, actor-critic)
- 🔬 Optimización científica y numérica
- 📈 Cualquier problema de minimización diferenciable

**Este notebook es tu referencia permanente** - vuelve a él cuando necesites:
- Elegir un optimizador
- Debuggear convergencia
- Entender papers de optimización
- Implementar variantes custom

---

[⬆ Volver a Tabla de Contenidos](#toc)

<a name='10'></a>

---
## ➡️ 10. Navegación

<div align="center">

### 🎉 ¡Has completado Gradient Descent! 🎉

**Continúa tu viaje en Machine Learning:**

---

### 📍 Ruta de ML Clásico

| Notebook | Status | Descripción |
|----------|--------|-------------|
| [01. Regresión Lineal](01-regresion-lineal.ipynb) | ✅ Completado | Fundamentos de regresión |
| [**02. Gradient Descent**](02-gradient-descent.ipynb) | ✅ **Estás aquí** | **Optimización** |
| [03. Regresión Logística](03-regresion-logistica.ipynb) | ⏭️ Siguiente | Clasificación binaria |
| [04. Clasificación Multiclase](04-clasificacion-multiclase.ipynb) | 🔜 Próximo | Softmax, one-vs-all |
| [05. Regularización](05-regularizacion.ipynb) | 🔜 Próximo | L1, L2, Elastic Net |

---

### 🔗 Enlaces Rápidos

- [← Anterior: 01. Regresión Lineal](01-regresion-lineal.ipynb)
- [→ Siguiente: 03. Regresión Logística](03-regresion-logistica.ipynb)
- [↑ Índice de ML Clásico](README.md)
- [🏠 Inicio de Tutoriales](../../README.md)

---

### 📚 Recursos Adicionales

**Si quieres profundizar más en Gradient Descent:**

1. **Teoría:**
   - Deep Learning Book (Goodfellow) - [Capítulo 8](https://www.deeplearningbook.org/contents/optimization.html)
   - Bottou et al. (2018) - Survey completo de 100+ páginas
   - Boyd & Vandenberghe - Convex Optimization

2. **Visualizaciones:**
   - [Distill.pub - Momentum](https://distill.pub/2017/momentum/)
   - [CS231n - Optimization](http://cs231n.github.io/optimization-1/)
   - [Loss Landscape Visualizer](https://losslandscape.com/)

3. **Papers:**
   - Ver **Sección 8** de este notebook (20+ papers curados)
   - [Papers with Code - Optimization](https://paperswithcode.com/methods/category/optimization)

4. **Implementaciones:**
   - [PyTorch Optimizers Source](https://github.com/pytorch/pytorch/tree/master/torch/optim)
   - [TensorFlow Optimizers](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers)
   - [JAX Optax](https://github.com/deepmind/optax)

5. **Cursos:**
   - Andrew Ng - Machine Learning (Coursera)
   - CS231n - Stanford (Computer Vision)
   - Fast.ai - Practical Deep Learning
   - CS224n - Stanford (NLP)

6. **Libros:**
   - Goodfellow et al. - Deep Learning
   - Bishop - Pattern Recognition and ML
   - Murphy - Probabilistic ML

---

### 📊 Tu Progreso

```
Ruta de ML Clásico:
[████████░░] 40% completado

✅ 01. Regresión Lineal
✅ 02. Gradient Descent  ← Estás aquí (8/8 ejercicios completados!)
⬜ 03. Regresión Logística
⬜ 04. Clasificación Multiclase
⬜ 05. Regularización
⬜ 06. SVM
⬜ 07. Árboles de Decisión
⬜ 08. Ensemble Methods
⬜ 09. Clustering
⬜ 10. PCA
```

---

### 💬 Feedback y Contribuciones

- 🐛 **Encontraste un bug?** Abre un issue
- 💡 **Tienes una sugerencia?** Pull request bienvenido
- ⭐ **Te gustó el notebook?** Dale una estrella al repo
- 📧 **Contacto:** feedback@tutorials-ai.com

---

### 🎓 Certificado de Completitud

Si completaste los 8 ejercicios con 70+ puntos:

```
╔════════════════════════════════════════════════════════════╗
║                                                            ║
║        CERTIFICADO DE COMPLETITUD                          ║
║                                                            ║
║        Gradient Descent: El Corazón del ML                 ║
║                                                            ║
║        Has dominado:                                       ║
║        ✓ Teoría fundamental de optimización                ║
║        ✓ Implementación de 8 funciones core                ║
║        ✓ Optimizadores avanzados (Momentum, Adam)          ║
║        ✓ 20+ papers fundamentales                          ║
║                                                            ║
║        Puntos obtenidos: ___/100                           ║
║                                                            ║
║        ¡Felicitaciones!                                    ║
║                                                            ║
╚════════════════════════════════════════════════════════════╝
```

---

**¡Nos vemos en el siguiente notebook! 🚀**

</div>

---

[⬆ Volver al inicio](#toc)